In [1]:
from pathlib import Path
import sys,json,hashlib,importlib.util
ROOT=Path("/home/featurize/work/biological_prior_validation").resolve()
RUN=ROOT/"data/splits/step2_q05_seed20261008"
PROTOCOL="task2_five_methods"
EXPERIMENT_NAME="task2_ordered_controls_num_v2"
NUMERICAL_AMENDMENT='adaptive_budget_1000_3000_v1'
RUN_ALL15=True 
BATCH_CELLS=192
if not ROOT.is_dir():raise FileNotFoundError(f"Please modify ROOT：{ROOT}")


In [2]:
def load_module(name,path):
    if not path.is_file():raise FileNotFoundError(f"Missing preceding source code: {path}, please complete the previous notebook first")
    spec=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
    return module
mc=load_module("randomized_controls",ROOT/"src/randomized_controls.py")
ps=load_module("parameter_search",ROOT/"src/parameter_search.py")
ev=load_module("evaluation",ROOT/"src/evaluation.py")


In [3]:
CORE_SOURCE = '"""Unchanged Step5 construction/scoring helpers; raw Galerkin audit added."""\nimport numpy as np\nimport scipy.sparse as sp\nfrom scipy.special import xlogy\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.cluster import MiniBatchKMeans\nfrom sklearn.neighbors import NearestNeighbors\ntry:import faiss\nexcept ImportError:faiss=None\ndef representation(Y,c,seed):\n n,g=Y.shape\n if n<max(3,c[\'min_metacells\']) or g<2:raise ValueError(\'Insufficient cells/genes\')\n svd=TruncatedSVD(n_components=min(c[\'svd_components\'],n-1,g-1),n_iter=5,n_oversamples=10,random_state=seed)\n Z=svd.fit_transform(Y).astype(np.float32);norm=np.linalg.norm(Z,axis=1,keepdims=True);Z=np.divide(Z,norm,out=np.zeros_like(Z),where=norm>0)\n requested=int(np.clip(round(n/c[\'target_cells_per_metacell\']),c[\'min_metacells\'],c[\'max_metacells\']))\n for factor in [1.,.85,.70,.55]:\n  k=min(n,max(c[\'min_metacells\'],round(requested*factor)))\n  km=MiniBatchKMeans(n_clusters=k,batch_size=min(8192,max(1024,n//5)),random_state=seed,n_init=3,reassignment_ratio=.01)\n  labels=km.fit_predict(Z);used,labels=np.unique(labels,return_inverse=True);counts=np.bincount(labels)\n  if len(counts)>=2 and counts.min()>=c[\'min_members\']:break\n else:raise ValueError(\'No metacell assignment satisfies pre-fixed minimum size\')\n k=min(c[\'knn_k\'],n-1);search=min(n,k+8)\n if faiss is not None:\n  faiss.omp_set_num_threads(1);idx=faiss.IndexHNSWFlat(Z.shape[1],32,faiss.METRIC_INNER_PRODUCT);idx.hnsw.efConstruction=80;idx.hnsw.efSearch=64;idx.add(np.ascontiguousarray(Z));similar,neighbors=idx.search(np.ascontiguousarray(Z),search);backend=\'FAISS HNSW, one thread\'\n else:\n  if n>10000:raise RuntimeError(\'Install faiss-cpu in the active kernel; refusing quadratic brute-force on a large context\')\n  nn=NearestNeighbors(n_neighbors=search,metric=\'cosine\',algorithm=\'brute\',n_jobs=1).fit(Z);dist,neighbors=nn.kneighbors(Z);similar=1-dist;backend=\'sklearn exact cosine, small data only\'\n nonself=neighbors!=np.arange(n)[:,None];select=nonself & (np.cumsum(nonself,axis=1)<=k)\n if not (select.sum(axis=1)==k).all():raise ValueError(\'kNN neighbor count mismatch\')\n W=sp.csr_matrix((np.clip(similar[select],0,1),(np.repeat(np.arange(n),k),neighbors[select])),shape=(n,n));W=W.maximum(W.T);W.setdiag(0);W.eliminate_zeros()\n P=sp.csr_matrix((np.ones(n),(np.arange(n),labels)),shape=(n,len(counts)))\n wm=(P.T@W@P).toarray();wm=(wm+wm.T)/2;np.fill_diagonal(wm,0);lc=np.diag(wm.sum(1))-wm\n inv=1/np.sqrt(counts);lc=inv[:,None]*lc*inv[None,:];vals,U=np.linalg.eigh(lc)\n if vals.min()<-1e-7 or vals.max()<=0:raise ValueError(\'Degenerate/non-PSD Galerkin cell operator\')\n raw_max=float(vals.max());raw_lc=lc.copy();raw_affinity=wm.copy();vals=np.maximum(vals,0)/raw_max;mean=(P.T@Y).toarray()/counts[:,None]\n C=U.T@(np.sqrt(counts)[:,None]*mean)\n return {\'labels\':labels,\'counts\':counts,\'U\':U,\'lc\':vals,\'C\':C,\'backend\':backend,\'svd_explained_variance\':float(np.nan_to_num(svd.explained_variance_ratio_).sum()),\'knn_edges\':int(W.nnz//2),\'raw_cell_operator\':raw_lc,\'raw_metacell_affinity\':raw_affinity,\'cell_scale\':raw_max,\'normalized_cell_operator\':raw_lc/raw_max}\n\ndef fit_fixed(r,ppi,alpha,beta):\n if beta==0:return r[\'C\']/(1+alpha*r[\'lc\'][:,None])\n q=r[\'C\']@ppi[\'V\'];den=1+alpha*r[\'lc\'][:,None]+beta*ppi[\'gamma\'][None,:]\n return (q/den)@ppi[\'V\'].T\n\ndef edge_energy(X,ppi):\n out=np.empty(len(ppi[\'u\']))\n for lo in range(0,len(out),2048):\n  diff=X[:,ppi[\'u\'][lo:lo+2048]]-X[:,ppi[\'v\'][lo:lo+2048]];out[lo:lo+2048]=np.einsum(\'ij,ij->j\',diff,diff)\n return out\n\ndef metacell_prediction(r,X):return (r[\'U\']@X)/np.sqrt(r[\'counts\'])[:,None]\n\nclass DevelopmentLoss:\n """Exact per-cell Poisson loss by sufficient statistics, with floor-boundary fallback."""\n def __init__(self,V,library,labels,q,scale,floor):\n  self.V=V.tocsr().astype(float);self.lib=np.asarray(library,float);self.labels=np.asarray(labels,int);self.q=q;self.scale=scale;self.floor=floor\n  if (self.lib<=0).any() or not 0<q<1 or floor<=0:raise ValueError(\'Invalid development exposure\')\n  m=int(labels.max())+1;n,g=V.shape;self.n_pairs=n*g\n  P=sp.csr_matrix((np.ones(n),(labels,np.arange(n))),shape=(m,n))\n  self.bs=(P@self.V).toarray();blog=self.V.copy();blog.data=xlogy(blog.data,blog.data)\n  self.blog=(P@blog).toarray();self.blib=(P@(sp.diags(np.log(self.lib))@self.V)).toarray()\n  self.exposure=np.bincount(labels,weights=self.lib,minlength=m);self.n=np.bincount(labels,minlength=m)\n  self.minimum=np.array([self.lib[labels==j].min() for j in range(m)]);self.maximum=np.array([self.lib[labels==j].max() for j in range(m)])\n def __call__(self,pred):\n  pred=np.asarray(pred,float)\n  if pred.shape!=self.bs.shape or not np.isfinite(pred).all():raise ValueError(\'Invalid development prediction\')\n  with np.errstate(over=\'raise\'):rate=np.expm1(np.maximum(pred,0))/self.scale*(1-self.q)/self.q\n  high=rate*self.minimum[:,None]>=self.floor;low=rate*self.maximum[:,None]<=self.floor;mix=~(high|low)\n  result=np.zeros_like(rate)\n  result[high]=(2*(self.blog-self.blib-self.bs*xlogy(np.ones_like(rate),np.maximum(rate,np.finfo(float).tiny))-self.bs+rate*self.exposure[:,None]))[high]\n  result[low]=(2*(self.blog-self.bs*np.log(self.floor)-self.bs+self.n[:,None]*self.floor))[low]\n  for group in np.unique(np.where(mix)[0]):\n   cols=np.where(mix[group])[0];rows=np.where(self.labels==group)[0];b=self.V[rows][:,cols].toarray();mu=np.maximum(self.lib[rows,None]*rate[group,cols][None,:],self.floor)\n   result[group,cols]=2*np.sum(xlogy(b,b/mu)-b+mu,axis=0)\n  value=float(result.sum()/self.n_pairs)\n  if not np.isfinite(value) or value<-1e-7:raise ValueError(\'Invalid development loss\')\n  return max(value,0)\n\ndef norm_counts(a,library,scale):\n y=(sp.diags(scale/library)@a.astype(float)).tocsr();y.data=np.log1p(y.data);return y\n'
SOURCE = '"""Task2.3 ordered exploratory experiments. Final target has no fitting API.\nImmutable inputs/cache receipts, explicit history of previously accessed B.\n"""\nfrom pathlib import Path\nimport json,hashlib,time,sys,platform,zipfile\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom scipy.special import kl_div\nfrom scipy.sparse.linalg import LinearOperator,cg\nimport randomized_controls as mc\nimport parameter_search as ps\nimport task2_core as core\nimport evaluation as ev\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nSTAGES={\'fixed\':[\'M0\',\'M1\'],\'adaptive\':[\'M2\'],\'matched\':[\'M3\'],\'rewired\':[\'M4\']}\nMETHODS=[\'M0\',\'M1\',\'M2\',\'M3\',\'M4\']\nNUMS=[\'deviance_sum\',\'normalized_sse_sum\',\'n_pairs\',\'n_cells\',\'mu_floor_pairs\',\'negative_log_predictions\']\ndef numerical_grid_amendment(grid):\n updated=json.loads(json.dumps(grid))\n if updated[\'precision\'][\'standard\'][\'adaptive_max_outer\']!=100 or updated[\'precision\'][\'tight\'][\'adaptive_max_outer\']!=300 or updated[\'source_coarse\'][\'convergence\'][\'adaptive_max_outer\']!=100:raise ValueError(\'Unexpected original numerical budgets\')\n updated[\'precision\'][\'standard\'][\'adaptive_max_outer\']=1000\n updated[\'precision\'][\'tight\'][\'adaptive_max_outer\']=3000\n updated[\'source_coarse\'][\'convergence\'][\'adaptive_max_outer\']=1000\n return updated\ndef numerical_accuracy_amendment(grid):\n updated=numerical_grid_amendment(grid)\n updated[\'precision\'][\'standard\'].update(cg_rtol=1e-10,cg_maxiter=3000,adaptive_x_rtol=1e-9,adaptive_s_atol=1e-8)\n updated[\'precision\'][\'tight\'].update(cg_rtol=1e-12,cg_maxiter=6000,adaptive_x_rtol=1e-11,adaptive_s_atol=1e-10)\n updated[\'source_coarse\'][\'convergence\']=json.loads(json.dumps(updated[\'precision\'][\'standard\']))\n return updated\ndef verify_accuracy_evidence(parent,grid,scopes,parent_contract):\n folder=parent/\'numerical_diagnostics/accuracy_v3_candidate\'\n plan=js(folder/\'diagnostic_plan.json\');summary=js(folder/\'summary.json\');table=pd.read_csv(folder/\'probes.csv\')\n selected=[s for line,drug in mc.FOUR for rep in mc.REPS for s in scopes if (s[\'cell_line\'],s[\'drug\'],s[\'replicate\'])==(line,drug,rep)]\n expected_plan={\'role\':\'development_only_diagnostic\',\'final_B_used\':False,\'authorizes_search\':False,\'standard\':grid[\'precision\'][\'standard\'],\'tight\':grid[\'precision\'][\'tight\'],\'thresholds\':grid[\'precision\'][\'thresholds\'],\'scope_ids\':[s[\'scope_id\'] for s in selected],\'methods\':[\'M0\',\'M1\',\'M2\',\'M4\'],\'M4_seed_index\':0,\'probe_parameters\':grid[\'precision\'][\'probe_parameters\'],\'engine_sha256\':parent_contract[\'engine_sha256\']}\n if plan!=expected_plan or summary!={\'status\':\'PASS\',\'n_probes\':152,\'n_failed\':0,\'final_B_used\':False,\'authorizes_search\':False}:raise ValueError(\'Diagnostic plan/summary does not match the proposed numerical rules\')\n expected=[]\n for s in selected:\n  for method in expected_plan[\'methods\']:\n   for c in expected_plan[\'probe_parameters\']:\n    for rule in ([\'q0.5\',\'high\'] if method==\'M2\' and c[\'beta\']>0 else [\'fixed_limit\']):expected.append((s[\'scope_id\'],method,c[\'alpha\'],0. if method==\'M0\' else c[\'beta\'],rule))\n actual=list(table[[\'scope_id\',\'method\',\'alpha\',\'beta\',\'lambda_rule\']].itertuples(index=False,name=None))\n if len(expected)!=152 or Counter(actual)!=Counter(expected) or not table.status.eq(\'PASS\').all():raise ValueError(\'Incomplete/duplicate/failed diagnostic probes\')\n for s in selected:\n  rows=table[table.scope_id.eq(s[\'scope_id\'])]\n  if any(not rows[k].eq(s[k]).all() for k in [\'cell_line\',\'drug\',\'replicate\',\'n_cells\',\'n_pairs\']):raise ValueError(\'Diagnostic scope/metadata mismatch\')\n for key in [\'relative_prediction_l2\',\'relative_development_deviance\',\'absolute_s_max\']:\n  values=table.loc[table.method.eq(\'M2\'),key] if key==\'absolute_s_max\' else table[key]\n  if not np.isfinite(values.to_numpy(float)).all() or not values.ge(0).all() or not values.le(grid[\'precision\'][\'thresholds\'][key]).all():raise ValueError(\'Diagnostic precision threshold failed: \'+key)\n for level in [\'standard\',\'tight\']:\n  v=table[level+\'_iterations\'].to_numpy(float)\n  if not np.isfinite(v).all() or (v<0).any() or (v>grid[\'precision\'][level][\'adaptive_max_outer\']).any() or (v!=np.floor(v)).any():raise ValueError(\'Invalid diagnostic iteration counts\')\n return {\'folder\':str(folder),\'files\':[{\'name\':name,\'sha256\':sha(folder/name)} for name in [\'diagnostic_plan.json\',\'probes.csv\',\'summary.json\']],\'status\':\'152_DIAGNOSTIC_PROBES_VERIFIED\',\'fresh_real_gates_required\':True}\ndef sha(p):return mc.digest(p)\ndef js(p):return mc.js(p)\ndef save(p,o):return mc.save(p,o)\ndef freeze(p,o):return mc.freeze(p,o)\ndef npz_save(p,**arrays):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n t=p.with_name(p.stem+\'.tmp.npz\');np.savez_compressed(t,**arrays);t.replace(p)\ndef read_npz(p,names=None):\n with np.load(p,allow_pickle=False) as z:return {k:z[k] for k in (z.files if names is None else names)}\ndef receipt(folder,names,meta):\n obj={**meta,\'files\':[{\'name\':n,\'sha256\':sha(Path(folder)/n)} for n in names]};freeze(Path(folder)/\'receipt.json\',obj);return obj\ndef verified_receipt(folder):\n r=js(Path(folder)/\'receipt.json\')\n for f in r[\'files\']:\n  if sha(Path(folder)/f[\'name\'])!=f[\'sha256\']:raise ValueError(\'Cache changed: \'+str(Path(folder)/f[\'name\']))\n return r\ndef sparse(folder,name):\n r=js(Path(folder)/\'receipt.json\');h=next(f[\'sha256\'] for f in r[\'files\'] if f[\'name\']==name)\n if sha(Path(folder)/name)!=h:raise ValueError(\'Count shard changed: \'+str(Path(folder)/name))\n a=sp.load_npz(Path(folder)/name).tocsr()\n if a.dtype.kind not in \'iu\' or (a.data<0).any():raise ValueError(\'Nonnegative integer counts required\')\n return a,h\ndef eligible_mask(frame,scope,phase):\n label=\'inner_evaluation_eligible\' if phase==\'inner\' else \'evaluation_eligible\'\n m=frame[label].fillna(False).to_numpy(dtype=bool,copy=True)\n for c in [\'cell_line\',\'drug\',\'replicate\']:m &= frame[c].eq(scope[c]).fillna(False).to_numpy(dtype=bool)\n return m\ndef tag(method,seed=None):return method if seed is None else f\'{method}_seed{seed:03d}\'\n\ndef weighted_laplacian(p,w):\n # Endpoints were canonicalized/validated once when the frozen graph was built.\n w=np.asarray(w,float)\n if w.shape!=p[\'w\'].shape or not np.isfinite(w).all() or (w<0).any():raise ValueError(\'Invalid effective weights\')\n W=sp.csr_matrix((np.r_[w,w],(np.r_[p[\'u\'],p[\'v\']],np.r_[p[\'v\'],p[\'u\']])),shape=(p[\'n_genes\'],p[\'n_genes\']))\n return (sp.diags(np.asarray(W.sum(1)).ravel())-W).tocsr()\n\ndef warm_solve(C,lc,L,a,b,tol,X0):\n shape=C.shape;diagonal=1+a*lc;pre_diag=diagonal[:,None]+b*L.diagonal()[None,:]\n def mv(z):\n  x=z.reshape(shape);return (diagonal[:,None]*x+b*(L@x.T).T).ravel()\n op=LinearOperator((C.size,C.size),matvec=mv,dtype=float);pre=LinearOperator(op.shape,matvec=lambda z:z/pre_diag.ravel(),dtype=float)\n # Solve the correction residual: plain warm-start CG can stop immediately when\n # a small gene effect is masked by the large common-expression RHS component.\n initial=X0.ravel();rhs=C.ravel()-mv(initial)\n delta,info=cg(op,rhs,M=pre,rtol=tol[\'cg_rtol\'],atol=0,maxiter=tol[\'cg_maxiter\']);x=initial+delta\n residual=float(np.linalg.norm(mv(x)-C.ravel())/max(np.linalg.norm(C),1e-12))\n if info!=0 or residual>max(5*tol[\'cg_rtol\'],1e-12):raise RuntimeError(f\'Warm CG failed: {info=}, {residual=}\')\n return x.reshape(shape)\n\nclass Experiment:\n def __init__(self,root,run=None,protocol=\'task2_five_methods_v1\',name=\'task2_ordered_controls_v1\',batch_cells=192,numerical_amendment=None):\n  self.root=Path(root).resolve();self.run=Path(run or self.root/\'data/splits/step2_q05_seed20261008_v1\').resolve();self.protocol=protocol\n  if Path(name).name!=name or not name or name in [\'.\',\'..\'] or not self.run.is_relative_to(self.root/\'data/splits\'):raise ValueError(\'Invalid experiment/run path\')\n  if not isinstance(batch_cells,int) or batch_cells<1:raise ValueError(\'Invalid scoring batch\')\n  self.batch=batch_cells;self.old=self.run/\'task2_protocols\'/protocol;self.gridfolder=self.old/\'step2_parameter_grid\';self.out=self.run/\'task2_experiments\'/name;self.out.mkdir(parents=True,exist_ok=True)\n  self.grid=js(self.gridfolder/\'parameter_grid.yaml\');self.common=js(self.old/\'common_protocol.json\');self.oldid=js(self.old/\'protocol_identity.json\');self.config=self.oldid[\'config\']\n  for folder in [self.old,self.gridfolder]:\n   for f in js(folder/\'output_manifest.json\')[\'files\']:\n    if sha(folder/f[\'name\'])!=f[\'sha256\']:raise ValueError(\'Protocol manifest changed: \'+f[\'name\'])\n  if js(self.root/\'configs/parameter_grid.yaml\')!=self.grid:raise ValueError(\'Root grid not the reviewed complete grid\')\n  if self.grid[\'status\']!=\'complete_search_rules_frozen_BEFORE_REAL_BENCHMARK\':raise ValueError(\'Step2.2 required\')\n  gid=js(self.gridfolder/\'grid_identity.json\')\n  if ps.ENGINE_SHA!=gid[\'engine_sha256\'] or mc.ENGINE_SHA!=self.oldid[\'engine_sha256\']:raise ValueError(\'Frozen tool source changed\')\n  if gid[\'source_protocol_sha256\']!=sha(self.old/\'protocol_identity.json\') or self.grid[\'common_protocol_sha256\']!=sha(self.old/\'common_protocol.json\'):raise ValueError(\'Protocol linkage mismatch\')\n  self.dev=self.run/\'development\'/self.config[\'development_name\'];refs=self.oldid[\'references\']\n  for p,key in [(self.run/\'identity.json\',\'identity_sha256\'),(self.dev/\'identity.json\',\'step4_identity_sha256\'),(self.dev/\'cell_roles.parquet\',\'roles_sha256\'),(Path(self.common[\'outer_pair_contract\']),\'pair_sha256\'),(Path(self.common[\'metric_contract\']),\'metric_sha256\')]:\n   if sha(p)!=refs[key]:raise ValueError(\'Source changed: \'+str(p))\n  self.metric=js(self.common[\'metric_contract\'])\n  if ev.ENGINE_SHA!=self.metric[\'engine_sha256\']:raise ValueError(\'Step3 evaluator source changed; do not alter scoring\')\n  self.lock=self.run/\'evaluation/target_access_lock.json\';self.old_lock=sha(self.lock)\n  if self.old_lock!=self.oldid[\'target_history\'][\'target_access_lock_sha256\']:raise ValueError(\'Step5 lock changed\')\n  history=self.oldid[\'target_history\'];prior=Path(history[\'prior_summary\']).parent\n  for p,h in [(prior/\'step5_summary.json\',history[\'prior_summary_sha256\']),(prior/\'frozen_model_registry.json\',history[\'prior_registry_sha256\'])]:\n   if sha(p)!=h:raise ValueError(\'Step5 historical record changed\')\n  self.repconfig=self.grid[\'shared_representation\'][\'parameters\'];rp=prior/\'experiment_contract.json\'\n  if sha(rp)!=self.grid[\'shared_representation\'][\'source_sha256\']:raise ValueError(\'Shared construction source changed\')\n  self.roles=pd.read_parquet(self.dev/\'cell_roles.parquet\').sort_values(\'matrix_row\').reset_index(drop=True)\n  pair=js(self.common[\'outer_pair_contract\']);cp=self.run/\'A_preprocessing\'/pair[\'cells_file\']\n  if sha(cp)!=self.metric[\'cells_sha256\']:raise ValueError(\'Cell table changed\')\n  self.cells=pd.read_parquet(cp).sort_values(\'matrix_row\').reset_index(drop=True)\n  if not np.array_equal(self.cells.matrix_row,np.arange(len(self.cells))) or len(self.cells)!=len(self.roles) or not self.cells.cell_barcode.eq(self.roles.cell_barcode).all():raise ValueError(\'Cell row mapping changed\')\n  self.size=int(js(self.run/\'identity.json\')[\'cells_per_block\']);self.grefs={\'outer\':self.common[\'outer_gene_universe\'],**self.common[\'inner_gene_universe\']};self.genes={}\n  for label,ref in self.grefs.items():\n   if sha(Path(ref[\'path\']))!=ref[\'sha256\']:raise ValueError(\'Gene universe changed\')\n   g=pd.read_parquet(ref[\'path\']).sort_values(\'selected_col\').reset_index(drop=True)\n   if not np.array_equal(g.selected_col,np.arange(len(g))) or not g.gene_idx.is_unique or not g.gene_symbol.is_unique:raise ValueError(\'Gene universe/order invalid\')\n   self.genes[label]=g\n  ep=self.root/\'data/processed/ppi_annotation_reference_edges.parquet\'\n  if sha(ep)!=refs[\'ppi_reference_sha256\']:raise ValueError(\'PPI annotation changed\')\n  self.edges=pd.read_parquet(ep)\n  roster=pd.read_csv(self.old/\'all_context_roster.csv\');self.scopes=[]\n  for li,line in enumerate(mc.LINES):\n   for di,drug in enumerate(mc.DRUGS):\n    for ri,rep in enumerate(mc.REPS):\n     rows=roster.loc[roster.cell_line.eq(line)&roster.drug.eq(drug)&roster.replicate.eq(rep)]\n     if len(rows)!=1:raise ValueError(\'Complete roster required\')\n     rr=rows.iloc[0]\n     self.scopes.append({\'scope_id\':(li*len(mc.DRUGS)+di)*2+ri,\'cell_line\':line,\'drug\':drug,\'replicate\':rep,\'n_cells\':int(rr.n_cells),\'n_pairs\':int(rr.n_pairs)})\n  env={\'python\':sys.version,\'numpy\':np.__version__,\'scipy\':__import__(\'scipy\').__version__,\'sklearn\':__import__(\'sklearn\').__version__,\'faiss\':None if core.faiss is None else getattr(core.faiss,\'__version__\',\'present\')}\n  contract={\'kind\':\'task2_ordered_exploratory_controls_v1\',\'scope\':\'all15 predeclared; firstfour precedes expansion\',\'methods\':METHODS,\'initial_seeds\':20,\'grid_sha256\':sha(self.gridfolder/\'parameter_grid.yaml\'),\n   \'metric_sha256\':sha(Path(self.common[\'metric_contract\'])),\'step2_1_identity_sha256\':sha(self.old/\'protocol_identity.json\'),\'engine_sha256\':ENGINE_SHA,\'core_sha256\':sha(Path(core.__file__)),\n   \'evaluator_sha256\':ev.ENGINE_SHA,\'randomization_engine_sha256\':mc.ENGINE_SHA,\'search_engine_sha256\':ps.ENGINE_SHA,\'old_target_history\':history,\'all_scopes\':self.scopes,\'construction_seed_rule\':\'firstfour: Step5 seed+1000*four_index+10*rep_index; later: seed+50000+1000*context_index+10*rep_index\',\n   \'M4_phase_transfer\':\'same procedure/seed index; independently randomized inner/outer universes per frozen phase streams; does not assert same edge assignment across different gene universes\',\'stability\':\'numerical/computational reproducibility only, never require favorable effect direction\',\'environment\':env,\'batch_cells\':self.batch}\n  if numerical_amendment is not None:\n   if numerical_amendment==\'adaptive_accuracy_v3\':\n    if name in [\'task2_ordered_controls_v1\',\'task2_ordered_controls_num_v2\']:raise ValueError(\'Accuracy v3 requires a new experiment name\')\n    parent=self.run/\'task2_experiments/task2_ordered_controls_num_v2\';gate=parent/\'scopes/scope_07/precision/gate.json\';probes=gate.with_name(\'probes.csv\');parent_contract=js(parent/\'experiment_contract.json\')\n    if js(gate)[\'status\']!=\'FAIL\' or js(gate).get(\'final_B_used\') is not False:raise ValueError(\'Expected development-only failed v2 gate\')\n    verified_receipt(gate.parent)\n    if list(parent.glob(\'scores/*/*/target_access_ledger.json\')) or list(parent.glob(\'scores/*/*/scores.csv\')) or list(parent.glob(\'scopes/*/inner/tuning/*\')):raise ValueError(\'Parent already entered search/scoring\')\n    if js(parent/\'effective_parameter_grid.json\')!=numerical_grid_amendment(self.grid):raise ValueError(\'Unexpected parent v2 rules\')\n    if parent_contract.get(\'numerical_amendment\',{}).get(\'version\')!=\'adaptive_budget_1000_3000_v1\' or parent_contract[\'grid_sha256\']!=contract[\'grid_sha256\'] or parent_contract[\'metric_sha256\']!=contract[\'metric_sha256\'] or parent_contract[\'old_target_history\']!=contract[\'old_target_history\']:raise ValueError(\'Parent protocol/metric linkage mismatch\')\n    failed=pd.read_csv(probes);mask=failed.method.eq(\'M2\')&failed.alpha.eq(100)&failed.beta.eq(100)&failed.lambda_rule.eq(\'q0.5\')&failed.status.eq(\'FAIL\')\n    if not mask.any() or not failed.loc[mask,\'absolute_s_max\'].gt(self.grid[\'precision\'][\'thresholds\'][\'absolute_s_max\']).all():raise ValueError(\'Parent does not show the documented s precision failure\')\n    self.grid=numerical_accuracy_amendment(self.grid);evidence=verify_accuracy_evidence(parent,self.grid,self.scopes,parent_contract)\n    amendment={\'version\':numerical_amendment,\'parent_experiment\':str(parent),\'parent_contract_sha256\':sha(parent/\'experiment_contract.json\'),\'parent_gate_sha256\':sha(gate),\'parent_probes_sha256\':sha(probes),\'role\':\'development_only_numerical_amendment\',\'new_final_B_used_for_amendment\':False,\'evidence\':evidence,\'previous_rules\':js(parent/\'effective_parameter_grid.json\')[\'precision\'],\'effective_rules\':self.grid[\'precision\'],\'candidate_grids_thresholds_processing_metrics_seeds\':\'unchanged\',\'source_coarse_convergence\':\'synchronized with effective standard\',\'fresh_real_gates_required\':True}\n   else:\n    if numerical_amendment!=\'adaptive_budget_1000_3000_v1\' or name==\'task2_ordered_controls_v1\':raise ValueError(\'Numerical amendment requires its declared version and a new experiment name\')\n    parent=self.run/\'task2_experiments/task2_ordered_controls_v1\';gate=parent/\'scopes/scope_03/precision/gate.json\';probes=gate.with_name(\'probes.csv\')\n    if js(gate)[\'status\']!=\'FAIL\' or js(gate).get(\'final_B_used\') is not False:raise ValueError(\'Expected development-only failed parent gate\')\n    verified_receipt(gate.parent)\n    if list(parent.glob(\'scores/*/*/target_access_ledger.json\')) or list(parent.glob(\'scores/*/*/scores.csv\')) or list(parent.glob(\'scopes/*/inner/tuning/*\')):raise ValueError(\'Parent already entered search/scoring; this amendment is restricted to the pre-search failure\')\n    failed=pd.read_csv(probes);mask=(failed.method.eq(\'M2\')&failed.alpha.eq(100)&failed.beta.eq(100)&failed.lambda_rule.eq(\'q0.5\')&failed.status.eq(\'FAIL\'))\n    if not mask.any() or not failed.loc[mask,\'failure_reason\'].eq(\'Adaptive maximum iterations reached\').all():raise ValueError(\'Parent failure does not match the documented numerical cause\')\n    self.grid=numerical_grid_amendment(self.grid)\n    amendment={\'version\':numerical_amendment,\'parent_experiment\':str(parent),\'parent_contract_sha256\':sha(parent/\'experiment_contract.json\'),\'parent_gate_sha256\':sha(gate),\'parent_probes_sha256\':sha(probes),\'role\':\'development_only_numerical_amendment\',\'new_final_B_used_for_amendment\':False,\'changes\':{\'standard.adaptive_max_outer\':[100,1000],\'tight.adaptive_max_outer\':[300,3000],\'source_coarse.convergence.adaptive_max_outer\':[100,1000]},\'tolerances_thresholds_candidate_grids_processing_metrics_seeds\':\'unchanged\',\'motivation\':\'User-reported T/V diagnostic: original standard hit cap, original tight and extended fits converged in 219 iterations; extended comparison PASS. Fresh real gates required for every scope.\'}\n   contract.update(numerical_amendment=amendment,effective_grid_sha256=hashlib.sha256(json.dumps(self.grid,sort_keys=True).encode()).hexdigest())\n  freeze(self.out/\'experiment_contract.json\',contract);self.contract=contract\n  if numerical_amendment is not None:\n   freeze(self.out/\'effective_parameter_grid.json\',self.grid);freeze(self.out/\'numerical_amendment.json\',amendment)\n   if numerical_amendment==\'adaptive_accuracy_v3\':\n    destination=self.out/\'numerical_diagnostic_evidence\';destination.mkdir(parents=True,exist_ok=True)\n    for item in evidence[\'files\']:\n     source=Path(evidence[\'folder\'])/item[\'name\'];target=destination/item[\'name\']\n     if target.exists() and sha(target)!=item[\'sha256\']:raise ValueError(\'Frozen diagnostic evidence changed\')\n     if not target.exists():target.write_bytes(source.read_bytes())\n  save(self.out/\'environment.json\',env);self.genes[\'outer\'][[\'gene_idx\',\'gene_symbol\']].to_csv(self.out/\'outer_gene_ids.csv\',index=False)\n def first_four(self):\n  return [next(s for s in self.scopes if (s[\'cell_line\'],s[\'drug\'],s[\'replicate\'])==(line,drug,rep)) for line,drug in mc.FOUR for rep in mc.REPS]\n def check_lock(self):\n  if sha(self.lock)!=self.old_lock:raise ValueError(\'Old Step5 target lock changed\')\n def scope_dir(self,s):return self.out/\'scopes\'/f"scope_{s[\'scope_id\']:02d}"\n def counts(self,s,phase,target=False):\n  if phase not in [\'inner\',\'outer\'] or (target and phase!=\'inner\'):raise ValueError(\'Fitting loader cannot read final B\')\n  genes=self.genes[s[\'replicate\'] if phase==\'inner\' else \'outer\'];cols=genes.gene_col.to_numpy(int);mask=eligible_mask(self.roles,s,phase)\n  pieces=[];rows=[];libs=[];sources=[]\n  for block in range((len(mask)+self.size-1)//self.size):\n   lo=block*self.size;keep=mask[lo:lo+self.size]\n   if not keep.any():continue\n   folder=(self.dev if phase==\'inner\' else self.run)/f\'block_{block:05d}\';name=(\'A_validation.npz\' if target else \'A_train.npz\') if phase==\'inner\' else \'A_input.npz\'\n   a,h=sparse(folder,name)\n   if a.shape[0]!=len(keep) or len(cols)==0 or cols.max()>=a.shape[1]:raise ValueError(\'Count row/column mismatch\')\n   library=np.asarray(a.sum(1)).ravel();pieces.append(a[keep][:,cols]);rows.extend(np.arange(lo,lo+len(keep))[keep]);libs.extend(library[keep]);sources.append({\'path\':str(folder/name),\'sha256\':h})\n  if not pieces:raise ValueError(\'Empty scope\')\n  libs=np.asarray(libs,dtype=np.int64)\n  if not target and (libs<=0).any():raise ValueError(\'Positive input libraries required\')\n  return sp.vstack(pieces,format=\'csr\'),np.asarray(rows,np.int64),libs,sources\n def base_ppi(self,s,phase):\n  label=s[\'replicate\'] if phase==\'inner\' else \'outer\';genes=self.genes[label];folder=self.out/\'graph_cache\'/label\n  if (folder/\'receipt.json\').exists():verified_receipt(folder);z=read_npz(folder/\'base.npz\');return self.ppi_from(z)\n  folder.mkdir(parents=True,exist_ok=True);mapping={x:i for i,x in enumerate(genes.gene_symbol)}\n  e=self.edges.loc[self.edges.gene_a.isin(mapping)&self.edges.gene_b.isin(mapping)];uv=np.column_stack([e.gene_a.map(mapping),e.gene_b.map(mapping)]).astype(np.int64);raw=e.ppi_weight.to_numpy(float)\n  if (raw<=0).any():raise ValueError(\'Positive base weights required\')\n  uv,raw=mc.canonical_edges(uv,len(genes),raw);uv,w,audit=mc.normalize_base(uv,raw,len(genes))\n  oldaudit=js(self.gridfolder/f\'{label}_base_normalization.json\')\n  if not np.isclose(audit[\'scale\'],oldaudit[\'scale\'],rtol=1e-8) or audit[\'before\'][\'support_edges\']!=oldaudit[\'before\'][\'support_edges\']:raise ValueError(\'Reviewed graph audit mismatch\')\n  npz_save(folder/\'base.npz\',uv=uv,raw=raw,w=w,n_genes=np.array(len(genes)));save(folder/\'normalization.json\',audit)\n  receipt(folder,[\'base.npz\',\'normalization.json\'],{\'gene_sha256\':self.grefs[label][\'sha256\']})\n  return self.ppi_from(read_npz(folder/\'base.npz\'))\n @staticmethod\n def ppi_from(z,spectrum=False):\n  p={\'uv\':z[\'uv\'],\'u\':z[\'uv\'][:,0],\'v\':z[\'uv\'][:,1],\'w\':z[\'w\'],\'raw\':z.get(\'raw\',z[\'w\']),\'n_genes\':int(z[\'n_genes\'])}\n  p[\'L\']=mc.graph_laplacian(p[\'uv\'],p[\'w\'],p[\'n_genes\'])\n  if spectrum:\n   vals,V=np.linalg.eigh(p[\'L\'].toarray());p.update(gamma=np.maximum(vals,0),V=V)\n  return p\n def graph(self,s,phase,method,seed=None,adaptive_s=None):\n  if method in [\'M3\',\'M4\'] and (not isinstance(seed,int) or not 0<=seed<20):raise ValueError(\'This initial experiment uses exactly seed indices 0..19\')\n  base=self.base_ppi(s,phase)\n  if method in [\'M0\',\'M1\',\'M2\']:return base\n  folder=self.scope_dir(s)/phase/\'graphs\'/tag(method,seed)\n  if (folder/\'receipt.json\').exists():verified_receipt(folder);return self.ppi_from(read_npz(folder/\'graph.npz\'))\n  folder.mkdir(parents=True,exist_ok=True)\n  rng=mc.rng_for(self.config[\'master_seed\'],method,s[\'scope_id\'],seed,phase)\n  if method==\'M3\':\n   if adaptive_s is None:raise ValueError(\'M3 requires fitted M2 weights\')\n   uv,w,audit=mc.matched_permutation(base[\'uv\'],base[\'w\']*adaptive_s,base[\'n_genes\'],rng)\n  elif method==\'M4\':uv,w,audit=mc.rewired_control(base[\'uv\'],base[\'raw\'],base[\'n_genes\'],rng,self.config[\'swap_multiplier\'],self.config[\'max_attempts_per_swap\'])\n  else:raise ValueError(\'Unknown graph method\')\n  npz_save(folder/\'graph.npz\',uv=uv,w=w,n_genes=np.array(base[\'n_genes\']));save(folder/\'audit.json\',audit)\n  receipt(folder,[\'graph.npz\',\'audit.json\'],{\'entropy\':[self.config[\'master_seed\'],int(method[1]),s[\'scope_id\'],seed,0 if phase==\'inner\' else 1],\'phase\':phase})\n  return self.ppi_from(read_npz(folder/\'graph.npz\'))\n def representation(self,s,phase):\n  folder=self.scope_dir(s)/phase/\'representation\'\n  if (folder/\'receipt.json\').exists():\n   r=verified_receipt(folder)\n   for item in r[\'input_sources\']:\n    if sha(Path(item[\'path\']))!=item[\'sha256\']:raise ValueError(\'Representation input changed\')\n   return read_npz(folder/\'representation.npz\')\n  a,rows,lib,sources=self.counts(s,phase);Y=core.norm_counts(a,lib,self.metric[\'normalization_scale\'])\n  context=(s[\'cell_line\'],s[\'drug\']);ri=mc.REPS.index(s[\'replicate\']);ix=mc.FOUR.index(context) if context in mc.FOUR else s[\'scope_id\']//2\n  seed=self.repconfig[\'seed\']+(1000*ix if context in mc.FOUR else 50000+1000*ix)+10*ri\n  r=core.representation(Y,self.repconfig,seed);folder.mkdir(parents=True,exist_ok=True)\n  raw=r[\'raw_cell_operator\'];aff=r[\'raw_metacell_affinity\'];scale=r[\'cell_scale\'];weighted=np.asarray(aff.sum(1)).ravel();degree=np.count_nonzero(aff,axis=1)\n  audit={\'source\':\'T only\' if phase==\'inner\' else \'full A only\',\'n_cells\':len(rows),\'n_genes\':a.shape[1],\'n_metacells\':len(r[\'counts\']),\'construction_seed\':seed,\'backend\':r[\'backend\'],\'knn_edges\':r[\'knn_edges\'],\n   \'normalization\':\'whiten membership first, then divide Galerkin Lc by own maximum eigenvalue\',\'cell_scale\':scale,\'raw_lambda_max\':float(np.linalg.eigvalsh(raw).max()),\'normalized_lambda_max\':float(r[\'lc\'].max()),\n   \'affinity_degree_before\':degree.tolist(),\'affinity_degree_after\':degree.tolist(),\'affinity_weighted_degree_before\':weighted.tolist(),\'affinity_weighted_degree_after\':(weighted/scale).tolist(),\n   \'affinity_total_weight_before\':float(aff.sum()/2),\'affinity_total_weight_after\':float(aff.sum()/2/scale),\'Galerkin_diagonal_before\':np.diag(raw).tolist(),\'Galerkin_diagonal_after\':(np.diag(raw)/scale).tolist(),\n   \'note\':\'whitened Galerkin operator is not ordinary affinity D-W; report affinity degree and actual operator diagonal separately\',\'within_metacell_affinity_removed\':True}\n  z={k:r[k] for k in [\'C\',\'lc\',\'U\',\'counts\',\'labels\',\'raw_cell_operator\',\'raw_metacell_affinity\',\'normalized_cell_operator\']};z.update(matrix_row=rows,library=lib,gene_idx=self.genes[s[\'replicate\'] if phase==\'inner\' else \'outer\'].gene_idx.to_numpy())\n  npz_save(folder/\'representation.npz\',**z);save(folder/\'cell_graph_audit.json\',audit)\n  receipt(folder,[\'representation.npz\',\'cell_graph_audit.json\'],{\'phase\':phase,\'input_sources\':sources,\'gene_sha256\':self.grefs[s[\'replicate\'] if phase==\'inner\' else \'outer\'][\'sha256\'],\'matrix_row_sha256\':hashlib.sha256(rows.tobytes()).hexdigest()})\n  return z\n def loss(self,s,r):\n  V,rows,_,sources=self.counts(s,\'inner\',target=True)\n  if not np.array_equal(rows,r[\'matrix_row\']):raise ValueError(\'T/V common-row mismatch\')\n  freeze(self.scope_dir(s)/\'inner/development_target_sources.json\',{\'role\':\'development_only\',\'sources\':sources})\n  return core.DevelopmentLoss(V,r[\'library\'],r[\'labels\'],self.common[\'scoring\'][\'inner_q\'],self.metric[\'normalization_scale\'],self.metric[\'mu_floor\'])\n def fit(self,r,p,method,c,tolerances,allow_degenerate_fixed=False):\n  a,b=c[\'alpha\'],c[\'beta\'];rule=c[\'lambda_rule\'];X=self.fixed(r,p,a,b,tolerances)\n  ss=np.ones(len(p[\'w\']));tau=\'fixed_limit\';value=\'fixed_limit\';iterations=0\n  if method==\'M2\' and b>0 and rule!=\'fixed_limit\':\n   energies=p.setdefault(\'cell_only_energy_cache\',{})\n   if a not in energies:energies[a]=core.edge_energy(r[\'C\']/(1+a*r[\'lc\'][:,None]),p)\n   energy=energies[a]\n   tau=ps.threshold(rule,energy,self.grid[\'source_coarse\'][\'lambda_rules\'])\n   if tau==\'DEGENERATE_NO_POSITIVE_ENERGY\':\n    if not allow_degenerate_fixed:return {\'status\':\'SKIPPED_DEGENERATE\',\'reason\':\'no positive T/A edge energy\'}\n    # Frozen degeneracy rule: input-only fixed limit, not a new target-driven selection.\n    return {\'status\':\'OK\',\'X\':X,\'prediction\':core.metacell_prediction(r,X),\'s\':ss,\'tau\':\'fixed_limit\',\'lambda_value\':\'fixed_limit\',\'iterations\':0,\'degenerate_fixed_limit\':True}\n   value=mc.lambda_from_threshold(b,tau);ss=np.clip(1-core.edge_energy(X,p)/tau,0,1)\n   for iterations in range(1,tolerances[\'adaptive_max_outer\']+1):\n    L=weighted_laplacian(p,p[\'w\']*ss);new=warm_solve(r[\'C\'],r[\'lc\'],L,a,b,tolerances,X)\n    sn=np.clip(1-core.edge_energy(new,p)/tau,0,1);dx=float(np.linalg.norm(new-X)/max(np.linalg.norm(X),1e-12));ds=float(np.max(np.abs(sn-ss),initial=0));X=new;ss=sn\n    if dx<=tolerances[\'adaptive_x_rtol\'] and ds<=tolerances[\'adaptive_s_atol\']:break\n   else:raise RuntimeError(\'Adaptive maximum iterations reached\')\n  elif b==0:value=1.;tau=\'inactive_beta0\'\n  pred=core.metacell_prediction(r,X)\n  if not np.isfinite(pred).all():raise ValueError(\'Nonfinite model prediction\')\n  return {\'status\':\'OK\',\'X\':X,\'prediction\':pred,\'s\':ss,\'tau\':tau,\'lambda_value\':value,\'iterations\':iterations}\n @staticmethod\n def fixed(r,p,a,b,tol):\n  if b==0:return r[\'C\']/(1+a*r[\'lc\'][:,None])\n  if \'V\' in p:\n   if \'rotated_C\' not in p:p[\'rotated_C\']=r[\'C\']@p[\'V\']\n   return (p[\'rotated_C\']/(1+a*r[\'lc\'][:,None]+b*p[\'gamma\'][None,:]))@p[\'V\'].T\n  cache=p.setdefault(\'fixed_initial_cache\',{});k=(a,b,tol[\'cg_rtol\'],tol[\'cg_maxiter\'])\n  if k not in cache:cache[k]=ps.solve(r[\'C\'],r[\'lc\'],p[\'L\'],a,b,tol[\'cg_rtol\'],tol[\'cg_maxiter\'])[0]\n  return cache[k]\n def precision_gate(self,s):\n  folder=self.scope_dir(s)/\'precision\';r=self.representation(s,\'inner\');base=self.base_ppi(s,\'inner\');loss=self.loss(s,r)\n  if (folder/\'receipt.json\').exists():\n   verified_receipt(folder)\n   if js(folder/\'gate.json\')[\'status\']!=\'PASS\':raise RuntimeError(\'Existing real precision gate failed; new numerical version required\')\n   return\n  folder.mkdir(parents=True,exist_ok=True);records=[];failure=False\n  try:m4=self.graph(s,\'inner\',\'M4\',0)\n  except (ValueError,RuntimeError) as e:\n   save(folder/\'gate.json\',{\'status\':\'FAIL\',\'reason\':str(e),\'role\':\'development_only\',\'final_B_used\':False});raise RuntimeError(\'M4 seed0 unavailable: \'+str(e))\n  for method in [\'M0\',\'M1\',\'M2\',\'M4\']:\n   for params in self.grid[\'precision\'][\'probe_parameters\']:\n    for rule in ([\'q0.5\',\'high\'] if method==\'M2\' and params[\'beta\']>0 else [\'fixed_limit\']):\n     c={**params,\'beta\':0. if method==\'M0\' else params[\'beta\'],\'lambda_rule\':rule};p=m4 if method==\'M4\' else base\n     row={\'method\':method,**c,\'target_role\':\'development_only\'}\n     try:\n      x=self.fit(r,p,method,c,self.grid[\'precision\'][\'standard\'],True);y=self.fit(r,p,method,c,self.grid[\'precision\'][\'tight\'],True)\n      if x[\'status\']!=\'OK\' or y[\'status\']!=\'OK\':raise RuntimeError(\'Degenerate/unavailable precision probe\')\n      comp=ps.precision_comparison(x[\'prediction\'],y[\'prediction\'],loss,self.grid[\'precision\'][\'thresholds\'],x[\'s\'] if method==\'M2\' else None,y[\'s\'] if method==\'M2\' else None)\n      row.update(comp,standard_tau=x[\'tau\'],tight_tau=y[\'tau\'],degenerate_training_fixed_limit=x.get(\'degenerate_fixed_limit\',False));failure|=comp[\'status\']!=\'PASS\'\n     except (RuntimeError,ValueError,FloatingPointError) as e:row.update(status=\'FAIL\',failure_reason=str(e));failure=True\n     records.append(row)\n  pd.DataFrame(records).to_csv(folder/\'probes.csv\',index=False);save(folder/\'gate.json\',{\'status\':\'FAIL\' if failure else \'PASS\',\'role\':\'development_only\',\'final_B_used\':False,\'n_probes\':len(records),\'representation_sha256\':sha(self.scope_dir(s)/\'inner/representation/representation.npz\')})\n  receipt(folder,[\'probes.csv\',\'gate.json\'],{\'scope_id\':s[\'scope_id\']})\n  if failure:raise RuntimeError(f"Precision gate failed for scope {s[\'scope_id\']}; see {folder/\'probes.csv\'}; do not run search or change grid from B")\n  print(\'Real T/V precision PASS:\',s[\'scope_id\'],flush=True)\n def prepare_precision(self,scopes=None):\n  for s in scopes or self.first_four():self.precision_gate(s)\n  self.check_lock()\n def tune(self,s,method,seed=None):\n  folder=self.scope_dir(s)/\'inner\'/\'tuning\'/tag(method,seed)\n  if (folder/\'receipt.json\').exists():verified_receipt(folder);return js(folder/\'selection.json\')\n  if js(self.scope_dir(s)/\'precision/gate.json\')[\'status\']!=\'PASS\':raise RuntimeError(\'Real precision gate required before tuning\')\n  r=self.representation(s,\'inner\');p=self.graph(s,\'inner\',method,seed);loss=self.loss(s,r);folder.mkdir(parents=True,exist_ok=True)\n  # Base spectral solve amortizes all fixed candidates on each graph.\n  if method in [\'M1\',\'M4\']:\n   vals,V=np.linalg.eigh(p[\'L\'].toarray());p.update(gamma=np.maximum(vals,0),V=V)\n  tol=self.grid[\'precision\'][\'standard\'];counter=0\n  def callback(c):\n   nonlocal counter\n   cp=folder/\'candidates\'/f"{hashlib.sha256(ps.candidate_id(c).encode()).hexdigest()}.json"\n   if cp.exists():return js(cp)\n   started=time.monotonic()\n   try:\n    fit=self.fit(r,p,method,c,tol)\n    record={\'status\':fit[\'status\'],\'target_role\':\'development_only\',\'development_deviance\':float(loss(fit[\'prediction\'])) if fit[\'status\']==\'OK\' else None,\'reason\':fit.get(\'reason\'),\n     \'tau\':fit.get(\'tau\'),\'lambda_value\':fit.get(\'lambda_value\'),\'iterations\':fit.get(\'iterations\'),\'elapsed_seconds\':time.monotonic()-started}\n   except (RuntimeError,ValueError,FloatingPointError) as e:record={\'status\':\'FAILED\',\'target_role\':\'development_only\',\'development_deviance\':None,\'failure_reason\':str(e),\'elapsed_seconds\':time.monotonic()-started}\n   freeze(cp,record);counter+=1\n   if counter%10==0:print(\'Development candidates:\',s[\'scope_id\'],tag(method,seed),counter,flush=True)\n   return record\n  search=ps.search(method,self.grid,callback);pd.DataFrame(search[\'rows\']).to_csv(folder/\'search_table.csv\',index=False)\n  par={k:search[\'selected\'][k] for k in [\'alpha\',\'beta\',\'lambda_rule\']};fit=self.fit(r,p,method,par,tol)\n  if fit[\'status\']!=\'OK\':raise RuntimeError(\'Selected development fit unavailable\')\n  selection={\'parameters\':par,\'development_deviance\':search[\'selected\'][\'development_deviance\'],\'unique_attempts\':search[\'unique_attempts\'],\'planned_slots\':search[\'planned_slots\'],\'coarse_boundary_flags\':search[\'coarse_boundary_flags\'],\'final_boundary_flags\':search[\'final_boundary_flags\'],\n   \'target_role\':\'development_only\',\'T_tau\':fit[\'tau\'],\'T_lambda_value\':fit[\'lambda_value\'],\'n_pairs\':loss.n_pairs}\n  save(folder/\'selection.json\',selection);npz_save(folder/\'selected_fit.npz\',X=fit[\'X\'],s=fit[\'s\'],prediction=fit[\'prediction\'])\n  receipt(folder,[\'selection.json\',\'search_table.csv\',\'selected_fit.npz\'],{\'method\':method,\'seed_index\':seed,\'Lc_sha256\':sha(self.scope_dir(s)/\'inner/representation/representation.npz\'),\'gene_sha256\':self.grefs[s[\'replicate\']][\'sha256\']})\n  return selection\n def model(self,s,method,seed=None):\n  if method not in METHODS or (method in [\'M3\',\'M4\'] and (not isinstance(seed,int) or not 0<=seed<20)) or (method not in [\'M3\',\'M4\'] and seed is not None):raise ValueError(\'Invalid method/initial seed index\')\n  folder=self.scope_dir(s)/\'outer\'/\'models\'/tag(method,seed)\n  if (folder/\'receipt.json\').exists():return verified_receipt(folder)\n  if (folder/\'failure.json\').exists():return js(folder/\'failure.json\')\n  folder.mkdir(parents=True,exist_ok=True);tol=self.grid[\'precision\'][\'standard\']\n  try:\n   r=self.representation(s,\'outer\');base=self.base_ppi(s,\'outer\')\n   if method==\'M3\':\n    self.model(s,\'M2\');m2=read_npz(self.scope_dir(s)/\'outer/models/M2/model.npz\');selection=js(self.scope_dir(s)/\'inner/tuning/M2/selection.json\');p=self.graph(s,\'outer\',\'M3\',seed,m2[\'s\']);par=selection[\'parameters\']\n    fit=self.fit(r,p,\'M3\',par,tol);tau=m2[\'tau\'].item();lv=m2[\'lambda_value\'].item();fit.update(tau=tau,lambda_value=lv)\n    self.matched_development(s,seed)\n   else:\n    selection=self.tune(s,method,seed);par=selection[\'parameters\'];p=self.graph(s,\'outer\',method,seed);fit=self.fit(r,p,method,par,tol,True)\n   if fit[\'status\']!=\'OK\':raise RuntimeError(\'Selected final A-only fit degenerate\')\n   # Fixed inputs/parameters, no B: computational replay at tighter precision.\n   replay=self.fit(r,p,method,par,self.grid[\'precision\'][\'tight\'],True)\n   if replay[\'status\']!=\'OK\':raise RuntimeError(\'Final A-only replay unavailable\')\n   rel=float(np.linalg.norm(fit[\'prediction\']-replay[\'prediction\'])/max(np.linalg.norm(replay[\'prediction\']),1e-12))\n   ds=float(np.max(np.abs(fit[\'s\']-replay[\'s\']),initial=0)) if method==\'M2\' else None\n   if rel>self.grid[\'precision\'][\'thresholds\'][\'relative_prediction_l2\'] or (ds is not None and ds>self.grid[\'precision\'][\'thresholds\'][\'absolute_s_max\']):raise RuntimeError(\'Final input-only numerical replay failed\')\n   w=p[\'w\']*fit[\'s\'] if method==\'M2\' else p[\'w\'];audit=mc.graph_audit(p[\'uv\'],w,p[\'n_genes\']);edf=mc.conditional_edf(r[\'lc\'],mc.graph_laplacian(p[\'uv\'],w,p[\'n_genes\']),par[\'alpha\'],par[\'beta\'])\n   extra={}\n   if method==\'M3\':\n    reference=mc.conditional_edf(r[\'lc\'],mc.graph_laplacian(base[\'uv\'],base[\'w\']*m2[\'s\'],base[\'n_genes\']),par[\'alpha\'],par[\'beta\'])\n    extra={\'conditional_trace_H_reference\':reference,\'conditional_trace_H_delta\':edf-reference,\'conditional_trace_H_relative_percent\':100*(edf-reference)/reference,\'trace_Lg_reference\':float(2*np.sum(base[\'w\']*m2[\'s\']))}\n   z={\'prediction\':fit[\'prediction\'],\'X\':fit[\'X\'],\'s\':fit[\'s\'],\'matrix_row\':r[\'matrix_row\'],\'labels\':r[\'labels\'],\'library_A\':r[\'library\'],\'gene_idx\':r[\'gene_idx\'],\'uv\':p[\'uv\'],\'effective_w\':w,\'tau\':np.array(str(fit[\'tau\'])),\'lambda_value\':np.array(str(fit[\'lambda_value\']))}\n   npz_save(folder/\'model.npz\',**z);save(folder/\'graph_audit.json\',{\'effective_gene\':audit,\'conditional_trace_H\':edf,**extra,\'conditional_EDF_is_not_total_adaptive_DF\':True})\n   metadata={\'status\':\'OK\',\'method\':method,\'seed_index\':seed,\'scope\':s,\'parameters\':par,\'A_tau\':fit[\'tau\'],\'A_lambda_value\':fit[\'lambda_value\'],\'source\':\'full A only; parameters from T/V\',\'new_final_B_used_for_fit\':False,\n    \'Lc_sha256\':sha(self.scope_dir(s)/\'outer/representation/representation.npz\'),\'gene_sha256\':self.grefs[\'outer\'][\'sha256\'],\'common_I_row_sha256\':hashlib.sha256(r[\'matrix_row\'].tobytes()).hexdigest(),\n    \'n_cells\':len(r[\'matrix_row\']),\'n_pairs\':len(r[\'matrix_row\'])*len(r[\'gene_idx\']),\'degenerate_A_fixed_limit\':fit.get(\'degenerate_fixed_limit\',False),\'input_only_replay\':{\'status\':\'PASS\',\'relative_prediction_l2\':rel,\'absolute_s_max\':ds},\'inner_selection_sha256\':sha(self.scope_dir(s)/\'inner/tuning\'/tag(\'M2\' if method==\'M3\' else method,None if method==\'M3\' else seed)/\'selection.json\')}\n   if metadata[\'n_cells\']!=s[\'n_cells\'] or metadata[\'n_pairs\']!=s[\'n_pairs\']:raise ValueError(\'Outer common scope mismatch\')\n   return receipt(folder,[\'model.npz\',\'graph_audit.json\'],metadata)\n  except (RuntimeError,ValueError,FloatingPointError) as e:\n   record={\'status\':\'FAILED\',\'method\':method,\'seed_index\':seed,\'scope\':s,\'failure_reason\':str(e),\'n_cells\':s[\'n_cells\'],\'n_pairs\':s[\'n_pairs\']};freeze(folder/\'failure.json\',record)\n   if method in [\'M0\',\'M1\',\'M2\']:raise\n   print(\'Randomized seed FAILED, retained:\',s[\'scope_id\'],tag(method,seed),str(e),flush=True);return record\n def matched_development(self,s,seed):\n  folder=self.scope_dir(s)/\'inner/matched_checks\'/tag(\'M3\',seed)\n  if (folder/\'receipt.json\').exists():verified_receipt(folder);return\n  folder.mkdir(parents=True,exist_ok=True);r=self.representation(s,\'inner\');m2=read_npz(self.scope_dir(s)/\'inner/tuning/M2/selected_fit.npz\');par=js(self.scope_dir(s)/\'inner/tuning/M2/selection.json\')[\'parameters\']\n  p=self.graph(s,\'inner\',\'M3\',seed,m2[\'s\']);fit=self.fit(r,p,\'M3\',par,self.grid[\'precision\'][\'standard\']);loss=self.loss(s,r)\n  save(folder/\'development_score.json\',{\'development_deviance\':float(loss(fit[\'prediction\'])),\'selection_role\':\'NONE; never select permutation/seed\',\'parameters\':par})\n  receipt(folder,[\'development_score.json\'],{\'method\':\'M3\',\'seed_index\':seed,\'status\':\'OK\'})\n def build_stage(self,stage,scopes):\n  if stage not in STAGES:raise ValueError(\'Unknown stage\')\n  preceding={\'adaptive\':[\'M0\',\'M1\'],\'matched\':[\'M0\',\'M1\',\'M2\'],\'rewired\':[\'M0\',\'M1\',\'M2\',\'M3\']}.get(stage,[])\n  for s in scopes:\n   for m in preceding:\n    seeds=range(20) if m==\'M3\' else [None]\n    for seed in seeds:\n     p=self.scope_dir(s)/\'outer/models\'/tag(m,seed)\n     if not (p/\'receipt.json\').exists() and not (p/\'failure.json\').exists():raise ValueError(\'Fixed order requires preceding method: \'+str(p))\n   for method in STAGES[stage]:\n    for seed in (range(20) if method in [\'M3\',\'M4\'] else [None]):\n     self.model(s,method,seed);print(\'Frozen final A model:\',s[\'scope_id\'],tag(method,seed),flush=True)\n  self.check_lock()\n def score_stage(self,stage,scopes,label):\n  records=[]\n  for s in scopes:\n   for m in STAGES[stage]:\n    for seed in (range(20) if m in [\'M3\',\'M4\'] else [None]):\n     folder=self.scope_dir(s)/\'outer/models\'/tag(m,seed)\n     if (folder/\'receipt.json\').exists():r=verified_receipt(folder);status=\'OK\';h=sha(folder/\'receipt.json\')\n     elif (folder/\'failure.json\').exists():r=js(folder/\'failure.json\');status=\'FAILED\';h=sha(folder/\'failure.json\')\n     else:raise ValueError(\'Every planned model needs snapshot/status before B access\')\n     records.append({\'scope\':s,\'method\':m,\'seed_index\':seed,\'status\':status,\'path\':str(folder),\'receipt_sha256\':h})\n  folder=self.out/\'scores\'/label/stage;folder.mkdir(parents=True,exist_ok=True)\n  registry={\'stage\':stage,\'label\':label,\'experiment_contract_sha256\':sha(self.out/\'experiment_contract.json\'),\'metric_sha256\':sha(Path(self.common[\'metric_contract\'])),\'records\':records,\'old_B_already_accessed\':True,\'B_role\':\'evaluation only; no fit/selection callback\'}\n  freeze(folder/\'frozen_registry.json\',registry)\n  freeze(folder/\'target_access_ledger.json\',{\'registry_sha256\':sha(folder/\'frozen_registry.json\'),\'old_step5_lock_sha256\':self.old_lock,\'exploratory_reuse\':True,\'no_model_replacement\':True})\n  # Above registry+ledger exist BEFORE any B shard is read.\n  accum={};models={}\n  for item in records:\n   if item[\'status\']==\'OK\':\n    k=(item[\'scope\'][\'scope_id\'],tag(item[\'method\'],item[\'seed_index\']));models[k]=read_npz(Path(item[\'path\'])/\'model.npz\',[\'prediction\',\'matrix_row\',\'labels\',\'library_A\',\'gene_idx\']);accum[k]=np.zeros(len(NUMS),float)\n  selected=self.genes[\'outer\'].gene_col.to_numpy(int)\n  for block in range((len(self.cells)+self.size-1)//self.size):\n   lo=block*self.size;meta=self.cells.iloc[lo:lo+self.size];active=[s for s in scopes if eligible_mask(meta,s,\'outer\').any()]\n   if not active:continue\n   shard=self.run/f\'block_{block:05d}\';B,bh=sparse(shard,\'B_target.npz\');A,ah=sparse(shard,\'A_input.npz\')\n   if A.shape!=B.shape or A.shape[0]!=len(meta):raise ValueError(\'Final shard shape mismatch\')\n   checkpoint=folder/\'blocks\'/f\'block_{block:05d}.json\'\n   if checkpoint.exists():\n    saved=js(checkpoint)\n    if saved[\'B_sha256\']!=bh or saved[\'A_sha256\']!=ah or saved[\'registry_sha256\']!=sha(folder/\'frozen_registry.json\'):raise ValueError(\'Score checkpoint source changed\')\n    parts=saved[\'parts\']\n   else:\n    la=np.asarray(A.sum(1)).ravel();lb=np.asarray(B.sum(1)).ravel();parts=[]\n    for s in active:\n     local=np.flatnonzero(eligible_mask(meta,s,\'outer\'))\n     for start in range(0,len(local),self.batch):\n      rows=local[start:start+self.batch];globalrows=rows+lo\n      for item in [x for x in records if x[\'scope\'][\'scope_id\']==s[\'scope_id\'] and x[\'status\']==\'OK\']:\n       k=(s[\'scope_id\'],tag(item[\'method\'],item[\'seed_index\']));model=models[k];where=np.searchsorted(model[\'matrix_row\'],globalrows)\n       if (where>=len(model[\'matrix_row\'])).any() or not np.array_equal(model[\'matrix_row\'][where],globalrows) or not np.array_equal(model[\'gene_idx\'],self.genes[\'outer\'].gene_idx.to_numpy()) or not np.array_equal(model[\'library_A\'][where],la[rows]):raise ValueError(\'Prediction/target common-I mismatch\')\n       pred=model[\'prediction\'][model[\'labels\'][where]];scores=ev.score_rows(B[rows][:,selected],pred,la[rows],lb[rows],self.metric[\'q\'],self.metric[\'normalization_scale\'],self.metric[\'mu_floor\'])\n       sums=scores.sum(numeric_only=True).to_dict();sums[\'n_cells\']=len(rows)\n       parts.append({\'scope_id\':s[\'scope_id\'],\'tag\':k[1],\'sums\':{n:float(sums[n]) for n in NUMS}})\n    save(checkpoint,{\'A_sha256\':ah,\'B_sha256\':bh,\'registry_sha256\':sha(folder/\'frozen_registry.json\'),\'parts\':parts})\n   for p in parts:accum[(p[\'scope_id\'],p[\'tag\'])]+=np.array([p[\'sums\'][n] for n in NUMS])\n   print(\'Final B score:\',label,stage,block+1,flush=True)\n  table=[]\n  for item in records:\n   s=item[\'scope\'];row={**s,\'method\':item[\'method\'],\'seed_index\':item[\'seed_index\'],\'status\':item[\'status\'],\'scenario\':\'A\',\'replicate_interpretation\':\'label-stratified molecular holdout, not cross-biological-replicate direction\'}\n   if item[\'status\']==\'OK\':\n    sums=dict(zip(NUMS,accum[(s[\'scope_id\'],tag(item[\'method\'],item[\'seed_index\']))].tolist()))\n    if int(sums[\'n_cells\'])!=s[\'n_cells\'] or int(sums[\'n_pairs\'])!=s[\'n_pairs\']:raise ValueError(\'Incomplete scoring coverage\')\n    modelrec=js(Path(item[\'path\'])/\'receipt.json\');audit=js(Path(item[\'path\'])/\'graph_audit.json\')\n    row.update(sums,mean_poisson_deviance=sums[\'deviance_sum\']/sums[\'n_pairs\'],normalized_mse=sums[\'normalized_sse_sum\']/sums[\'n_pairs\'],**modelrec[\'parameters\'],A_tau=modelrec[\'A_tau\'],A_lambda_value=modelrec[\'A_lambda_value\'],\n     trace_Lg=audit[\'effective_gene\'][\'trace_Lg\'],conditional_trace_H=audit[\'conditional_trace_H\'],zero_weight_edges=audit[\'effective_gene\'][\'zero_weight_edges\'],weight_sum=audit[\'effective_gene\'][\'weight_sum\'],gene_sha256=modelrec[\'gene_sha256\'],Lc_sha256=modelrec[\'Lc_sha256\'],common_I_row_sha256=modelrec[\'common_I_row_sha256\'])\n    for field in [\'conditional_trace_H_reference\',\'conditional_trace_H_delta\',\'conditional_trace_H_relative_percent\',\'trace_Lg_reference\']:\n     if field in audit:row[field]=audit[field]\n   else:row[\'failure_reason\']=js(Path(item[\'path\'])/\'failure.json\')[\'failure_reason\']\n   table.append(row)\n  frame=pd.DataFrame(table);frame.to_csv(folder/\'scores.csv\',index=False);save(folder/\'score_receipt.json\',{\'scores_sha256\':sha(folder/\'scores.csv\'),\'registry_sha256\':sha(folder/\'frozen_registry.json\')});self.check_lock();return frame\n def run_stage(self,stage,scopes=None,label=\'first_four\'):\n  scopes=scopes or self.first_four();self.build_stage(stage,scopes);return self.score_stage(stage,scopes,label)\n def finalize_first_four(self):\n  scopes=self.first_four();frames=[];checks=[]\n  for stage in STAGES:\n   folder=self.out/\'scores/first_four\'/stage;rec=js(folder/\'score_receipt.json\')\n   if sha(folder/\'scores.csv\')!=rec[\'scores_sha256\']:raise ValueError(\'Score output changed\')\n   registry=js(folder/\'frozen_registry.json\')\n   for item in registry[\'records\']:\n    fn=\'receipt.json\' if item[\'status\']==\'OK\' else \'failure.json\'\n    if sha(Path(item[\'path\'])/fn)!=item[\'receipt_sha256\']:raise ValueError(\'Registered model snapshot changed\')\n   frames.append(pd.read_csv(folder/\'scores.csv\'))\n  table=pd.concat(frames,ignore_index=True);table.to_csv(self.out/\'first_four_all_scores.csv\',index=False)\n  stable=True\n  for s in scopes:\n   subset=table.loc[table.scope_id.eq(s[\'scope_id\'])]\n   if len(subset)!=43:raise ValueError(\'First-four null distribution not complete\')\n   if not subset.status.eq(\'OK\').all():stable=False\n   for method in METHODS:\n    count=int(subset.method.eq(method).sum());assert count==(20 if method in [\'M3\',\'M4\'] else 1)\n   verified_receipt(self.scope_dir(s)/\'precision\');gate=js(self.scope_dir(s)/\'precision/gate.json\')\n   if gate[\'status\']!=\'PASS\':stable=False\n   for item in subset.loc[subset.status.eq(\'OK\')].to_dict(\'records\'):\n    seed=None if pd.isna(item[\'seed_index\']) else int(item[\'seed_index\']);p=self.scope_dir(s)/\'outer/models\'/tag(item[\'method\'],seed);r=verified_receipt(p)\n    if r[\'input_only_replay\'][\'status\']!=\'PASS\':stable=False\n   checks.append({\'scope_id\':s[\'scope_id\'],\'all_status_OK\':bool(subset.status.eq(\'OK\').all()),\'rows\':len(subset),\'precision_PASS\':gate[\'status\']==\'PASS\'})\n  self.check_lock();state={\'status\':\'PASS\' if stable else \'BLOCKED_FAILED_RANDOM_SEED_OR_NUMERICS\',\'criteria\':\'technical reproducibility, source hashes, exact common-I and complete 20-seed distributions; never favorable effect direction\',\'scope_checks\':checks,\'all_scores_sha256\':sha(self.out/\'first_four_all_scores.csv\'),\'old_lock_unchanged\':True}\n  freeze(self.out/\'first_four_stability_gate.json\',state);return table,state\n def run_all15(self):\n  self.finalize_first_four()\n  if js(self.out/\'first_four_stability_gate.json\')[\'status\']!=\'PASS\':raise RuntimeError(\'Four-context stability gate must pass before expansion\')\n  self.prepare_precision(self.scopes)\n  for stage in STAGES:self.run_stage(stage,self.scopes,\'all15\')\n  frame=pd.concat([pd.read_csv(self.out/\'scores/all15\'/s/\'scores.csv\') for s in STAGES],ignore_index=True)\n  if len(frame)!=30*43:raise ValueError(\'All15 roster incomplete\')\n  frame.to_csv(self.out/\'all15_all_scores.csv\',index=False);save(self.out/\'all15_summary.json\',{\'status\':\'all15_exploratory_completed\',\'n_scopes\':30,\'n_rows\':len(frame),\'metrics_grid_processing_unchanged\':True,\'scored_all_contexts_together\':True,\'failed_random_seeds\':int(frame.status.ne(\'OK\').sum())});return frame\n def review_bundle(self):\n  scored=sum(int(pd.read_csv(p).status.eq(\'OK\').sum()) for p in self.out.glob(\'scores/*/*/scores.csv\'))\n  summary={\'status\':\'firstfour_completed\' if (self.out/\'first_four_stability_gate.json\').exists() else \'partial_run\',\'experiment_path\':str(self.out),\'full15_run\':(self.out/\'all15_summary.json\').exists(),\'new_final_B_scored\':scored>0,\'successful_score_rows_across_stage_outputs\':scored,\'stage_access_ledgers_created\':len(list(self.out.glob(\'scores/*/*/target_access_ledger.json\'))),\'previous_target_history_retained\':True,\'old_lock_before\':self.old_lock,\'old_lock_after\':sha(self.lock),\'scenario_B\':\'BLOCKED_PENDING_REPLICATE_PROVENANCE\',\'independent_biological_uncertainty\':\'not inferred from cell count\'}\n  save(self.out/\'task2_step3_summary.json\',summary)\n  files=[p for p in sorted(self.out.rglob(\'*\')) if p.is_file() and p.suffix in [\'.json\',\'.csv\'] and \'candidates\' not in p.parts and \'blocks\' not in p.parts and p.name!=\'review_manifest.json\']\n  save(self.out/\'review_manifest.json\',{\'files\':[{\'name\':str(p.relative_to(self.out)),\'sha256\':sha(p)} for p in files]+[{\'name\':\'configs/parameter_grid.yaml\',\'sha256\':sha(self.gridfolder/\'parameter_grid.yaml\')},{\'name\':\'src/task2_experiments.py\',\'sha256\':sha(Path(__file__))},{\'name\':\'src/task2_core.py\',\'sha256\':sha(Path(core.__file__))}]})\n  filename=f\'task2_step3_review_bundle_{self.out.name}.zip\' if \'numerical_amendment\' in self.contract else \'task2_step3_review_bundle.zip\'\n  path=self.root/\'results/summary\'/filename;path.parent.mkdir(parents=True,exist_ok=True)\n  with zipfile.ZipFile(path,\'w\',zipfile.ZIP_DEFLATED) as z:\n   for p in sorted(self.out.rglob(\'*\')):\n    if p.is_file() and p.suffix in [\'.json\',\'.csv\'] and \'candidates\' not in p.parts and \'blocks\' not in p.parts:z.write(p,str(p.relative_to(self.out)))\n   z.write(self.gridfolder/\'parameter_grid.yaml\',\'configs/parameter_grid.yaml\')\n   z.write(Path(__file__),\'src/task2_experiments.py\');z.write(Path(core.__file__),\'src/task2_core.py\')\n  return path\n'
def install_source(name,content):
    path=ROOT/"src"/(name+".py");path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.read_bytes()!=content.encode("utf-8"):
        old_sha=hashlib.sha256(path.read_bytes()).hexdigest()
        backup=path.with_name(name+"."+old_sha[:12]+".backup.py")
        if not backup.exists():backup.write_bytes(path.read_bytes())
    path.write_text(content,encoding="utf-8")
    return load_module(name,path)
core=install_source("task2_core",CORE_SOURCE)
te=install_source("task2_experiments",SOURCE)
print("Task2 experiment engine:",te.ENGINE_SHA)


Task2 experiment engine: 615b620a44fbb64aff8e38d9bf8a729050a680963f44f0147c4828b5365f9e2d


In [4]:
TEST_SOURCE = "from pathlib import Path\nimport sys,json,tempfile,contextlib,io,hashlib\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom scipy.special import kl_div\nfrom unittest.mock import patch\nimport task2_experiments as te\nimport randomized_controls as mc\nimport parameter_search as ps\nimport evaluation as ev\nimport task2_core as core\n\ndef reject(f):\n try:f()\n except (ValueError,RuntimeError,FileNotFoundError):return\n raise AssertionError('Expected rejection')\n\ndef fixture(root):\n run=root/'data/splits/step2_fixture';dev=run/'development/inner_q05_seed20261008_v1';prep=run/'A_preprocessing';prior=run/'step5/step5_four_contexts_v1'\n for p in [dev,prep,prior,run/'evaluation',root/'data/processed']:p.mkdir(parents=True,exist_ok=True)\n rng=np.random.default_rng(56);records=[]\n for l in mc.LINES:\n  for d in mc.DRUGS:\n   for rep in mc.REPS:\n    for i in range(8):records.append({'cell_line':l,'drug':d,'replicate':rep})\n records.append({'cell_line':pd.NA,'drug':pd.NA,'replicate':pd.NA});n=len(records);size=32\n cells=pd.DataFrame(records);cells['matrix_row']=np.arange(n);cells['cell_barcode']=[f'c{i}' for i in range(n)];cells['evaluation_eligible']=[True]*(n-1)+[False]\n for c in ['cell_line','drug','replicate']:cells[c]=cells[c].astype('string')\n genes=pd.DataFrame({'selected_col':range(8),'gene_idx':range(8),'gene_col':range(8),'gene_symbol':[f'g{i}' for i in range(8)]});cp=prep/'evaluation_cells.parquet';gp=prep/'selected_genes_A.parquet';cells.to_parquet(cp,index=False);genes.to_parquet(gp,index=False)\n pair={'cells_file':cp.name,'genes_file':gp.name,'cells_sha256':te.sha(cp),'genes_sha256':te.sha(gp),'n_cells':n-1,'n_genes':8,'n_pairs':(n-1)*8,'q':.5,'normalization_scale':10000}\n te.save(prep/'evaluation_pair_contract.json',pair);metric={**pair,'pair_contract_sha256':te.sha(prep/'evaluation_pair_contract.json'),'engine_sha256':ev.ENGINE_SHA,'mu_floor':1e-8,'relative_min_denominator':1e-12,'primary':'mean Poisson deviance','secondary':'normalized MSE'}\n te.save(run/'evaluation/metric_contract.json',metric);te.save(run/'identity.json',{'q':.5,'cells_per_block':size})\n roles=cells.copy();roles['inner_evaluation_eligible']=roles.evaluation_eligible;roles.to_parquet(dev/'cell_roles.parquet',index=False);inner=[]\n for rep in mc.REPS:\n  p=dev/f'{rep}_inner_selected_genes.parquet';genes.to_parquet(p,index=False);inner.append({'replicate':rep,'selected_genes_sha256':te.sha(p)})\n te.save(dev/'step4_summary.json',{'status':'step4_prepared','metric_contract_sha256_after':te.sha(run/'evaluation/metric_contract.json'),'inner_genes_by_replicate':inner});te.save(dev/'scenario_A_contract.json',{'inner_q':.5});te.save(dev/'identity.json',{'status':'prepared'})\n te.save(prior/'step5_summary.json',{'status':'four_context_baseline_scored','final_B_accessed':True,'contexts':mc.FOUR});te.save(prior/'frozen_model_registry.json',{'status':'frozen'});te.save(run/'evaluation/target_access_lock.json',{'kind':'previous_B_already_accessed'})\n construction={'seed':20261008,'svd_components':2,'knn_k':7,'target_cells_per_metacell':1050,'min_metacells':2,'max_metacells':2,'min_members':1};te.save(prior/'experiment_contract.json',{'config':construction})\n pd.DataFrame({'gene_a':[f'g{i}' for i in range(8)],'gene_b':[f'g{(i+1)%8}' for i in range(8)],'ppi_weight':np.linspace(.7,.9,8)}).to_parquet(root/'data/processed/ppi_annotation_reference_edges.parquet',index=False)\n T=rng.poisson(25,size=(n,8));V=rng.poisson(25,size=(n,8));B=rng.poisson(50,size=(n,8));A=T+V\n B[16,:]=0 # First Lapatinib scope: zero-library target row remains evaluated.\n for block in range((n+size-1)//size):\n  lo=block*size;hi=min(lo+size,n)\n  for folder,items in [(run/f'block_{block:05d}',{'A_input.npz':A[lo:hi],'B_target.npz':B[lo:hi]}),(dev/f'block_{block:05d}',{'A_train.npz':T[lo:hi],'A_validation.npz':V[lo:hi]})]:\n   folder.mkdir();files=[]\n   for name,a in items.items():sp.save_npz(folder/name,sp.csr_matrix(a));files.append({'name':name,'sha256':te.sha(folder/name)})\n   te.save(folder/'receipt.json',{'files':files})\n cfg=mc.default_config(root);cfg['run_path']=str(run);old,_=mc.prepare_contract(cfg);ps.prepare(root,run,cfg['protocol_name'],mc)\n return run\n\ndef mathematical_tests():\n rng=np.random.default_rng(88);C=rng.normal(size=(3,6));uv=np.array([[i,i+1] for i in range(5)]);L=mc.graph_laplacian(uv,np.linspace(.2,1,5),6);lc=np.array([0.,.3,1.])\n oracle=np.linalg.solve(np.kron(np.diag(1+2*lc),np.eye(6))+3*np.kron(np.eye(3),L.toarray()),C.ravel()).reshape(C.shape)\n x,_=ps.solve(C,lc,L,2.,3.);assert np.allclose(x,oracle,rtol=1e-7,atol=1e-8)\n p={'u':uv[:,0],'v':uv[:,1],'uv':uv,'w':np.linspace(.2,1,5),'n_genes':6}\n assert np.allclose(te.weighted_laplacian(p,p['w']).toarray(),L.toarray())\n tol={'cg_rtol':1e-8,'cg_maxiter':1000};warm=te.warm_solve(C,lc,L,2.,3.,tol,np.zeros_like(C));assert np.allclose(warm,oracle,rtol=1e-7,atol=1e-8)\n reject(lambda:te.warm_solve(C,lc,L,2.,3.,{'cg_rtol':1e-14,'cg_maxiter':1},np.zeros_like(C)))\n frame=pd.DataFrame({'evaluation_eligible':[True,False],'cell_line':pd.Series(['A172',pd.NA],dtype='string'),'drug':pd.Series(['Vehicle',pd.NA],dtype='string'),'replicate':pd.Series(['replicate_1',pd.NA],dtype='string')})\n mask=te.eligible_mask(frame,{'cell_line':'A172','drug':'Vehicle','replicate':'replicate_1'},'outer');assert mask.tolist()==[True,False]\n # Simulate pandas read-only views even on older pandas installations.\n original_to_numpy=pd.Series.to_numpy\n def readonly_view(series,*args,**kwargs):\n  arr=original_to_numpy(series,*args,**kwargs)\n  if not kwargs.get('copy',False):\n   arr=arr.view();arr.setflags(write=False)\n  return arr\n scope={'cell_line':'A172','drug':'Vehicle','replicate':'replicate_1'}\n nullable=frame.copy();nullable['evaluation_eligible']=pd.Series([True,pd.NA],dtype='boolean');nullable['inner_evaluation_eligible']=nullable.evaluation_eligible\n before=nullable.copy(deep=True)\n with patch.object(pd.Series,'to_numpy',readonly_view):\n  for phase in ['inner','outer']:\n   assert te.eligible_mask(nullable,scope,phase).tolist()==[True,False]\n   assert te.eligible_mask(nullable,{**scope,'drug':'no_match'},phase).tolist()==[False,False]\n   assert te.eligible_mask(nullable.iloc[:0],scope,phase).size==0\n pd.testing.assert_frame_equal(nullable,before)\n B=sp.csr_matrix([[0,2,0],[3,0,1],[0,0,0]]);library=np.array([1.,10.,3.]);labels=np.array([0,0,1]);pred=np.array([[1e-9,1.2,-.5],[0.,.7,2.]])\n for q in [.2,.5,.8]:\n  for floor in [1e-8,.1,1.]:\n   loss=core.DevelopmentLoss(B,library,labels,q,10.,floor);mu=np.maximum(np.expm1(np.maximum(pred[labels],0))*library[:,None]/10*(1-q)/q,floor)\n   assert np.isclose(loss(pred),np.mean(2*kl_div(B.toarray(),mu)),rtol=1e-10,atol=1e-10)\n return {'status':'PASS','checks':['CG/warm CG independent Kronecker oracle and fast Laplacian identity','read-only NumPy views: nullable metadata, inner/outer, no matches, empty frame, input preservation','development sufficient statistics vs dense all-zero/mixed-floor/q oracle']}\n\ndef integration_tests():\n with tempfile.TemporaryDirectory() as tmp:\n  root=Path(tmp);run=fixture(root);old_lock=te.sha(run/'evaluation/target_access_lock.json');ex=te.Experiment(root,run,batch_cells=3);s=ex.first_four()[0]\n  zero={'C':np.zeros((2,8)),'lc':np.array([0.,1.]),'U':np.eye(2),'counts':np.ones(2)};p=ex.base_ppi(s,'inner');par={'alpha':1.,'beta':1.,'lambda_rule':'q0.5'}\n  assert ex.fit(zero,p,'M2',par,ex.grid['precision']['standard'])['status']=='SKIPPED_DEGENERATE'\n  deg=ex.fit(zero,p,'M2',par,ex.grid['precision']['standard'],True);assert deg['status']=='OK' and deg['degenerate_fixed_limit'] and deg['tau']=='fixed_limit'\n  # Use exactly the frozen real grids and twenty seeds. One scope exercises the\n  # complete expensive path; remaining first-four/extension scope orchestration\n  # is tested separately with predeclared scope selection, not fabricated scores.\n  scopes=[s];log=io.StringIO();reads=[];original=sp.load_npz\n  def guard(path,*a,**kw):\n   name=Path(path).name\n   if name=='C_counts.npz':raise AssertionError('No C read')\n   if name=='B_target.npz':\n    stages=[p.parent.name for p in (ex.out/'scores/fixture').glob('*/target_access_ledger.json')]\n    assert stages,'B read before registry/ledger';reads.append(str(path))\n   return original(path,*a,**kw)\n  with contextlib.redirect_stdout(log),patch('scipy.sparse.load_npz',guard):\n   reject(lambda:ex.counts(s,'outer',target=True))\n   reject(lambda:ex.build_stage('adaptive',scopes))\n   ex.prepare_precision(scopes)\n   assert not reads,'Precision must not read B'\n   for stage in te.STAGES:\n    ex.build_stage(stage,scopes)\n    if stage=='fixed':assert not reads\n    table=ex.score_stage(stage,scopes,'fixture')\n    assert table.status.eq('OK').all(),table[['method','seed_index','status','failure_reason']] if 'failure_reason' in table else table\n    assert table.n_pairs.eq(s['n_pairs']).all()\n   combined=pd.concat([pd.read_csv(ex.out/'scores/fixture'/stage/'scores.csv') for stage in te.STAGES],ignore_index=True)\n   assert len(combined)==43 and combined.method.value_counts().to_dict()=={'M3':20,'M4':20,'M0':1,'M1':1,'M2':1}\n   assert combined.Lc_sha256.nunique()==combined.gene_sha256.nunique()==combined.common_I_row_sha256.nunique()==1\n   assert len(combined.loc[combined.method.eq('M3')])==20\n   m2trace=float(combined.loc[combined.method.eq('M2'),'trace_Lg'].iloc[0]);assert np.allclose(combined.loc[combined.method.eq('M3'),'trace_Lg'],m2trace)\n   # Independently verify final evaluator on a frozen model, including zero B library.\n   r=te.read_npz(ex.scope_dir(s)/'outer/models/M1/model.npz');allB=[];allA=[]\n   for block in range((len(ex.cells)+ex.size-1)//ex.size):\n    lo=block*ex.size;take=te.eligible_mask(ex.cells.iloc[lo:lo+ex.size],s,'outer')\n    if take.any():allB.append(original(run/f'block_{block:05d}/B_target.npz')[take].toarray());allA.append(original(run/f'block_{block:05d}/A_input.npz')[take].toarray())\n   B=np.vstack(allB);A=np.vstack(allA);pred=r['prediction'][r['labels']];mu=np.maximum(np.expm1(np.maximum(pred,0))*A.sum(1)[:,None]/10000,1e-8)\n   observed=float(combined.loc[combined.method.eq('M1'),'mean_poisson_deviance'].iloc[0]);assert np.isclose(observed,np.mean(2*kl_div(B,mu)),rtol=1e-10)\n   target=np.log1p(np.divide(10000*B,B.sum(1)[:,None],out=np.zeros_like(B,dtype=float),where=B.sum(1)[:,None]>0))\n   secondary=float(combined.loc[combined.method.eq('M1'),'normalized_mse'].iloc[0]);assert np.isclose(secondary,np.mean((pred-target)**2),rtol=1e-10)\n   before=te.sha(ex.scope_dir(s)/'outer/models/M2/model.npz');again=ex.score_stage('adaptive',scopes,'fixture');assert te.sha(ex.scope_dir(s)/'outer/models/M2/model.npz')==before\n   assert again.n_pairs.eq(s['n_pairs']).all();assert te.sha(run/'evaluation/target_access_lock.json')==old_lock\n   reject(lambda:ex.run_all15())\n   reject(lambda:ex.model(s,'M4',20))\n   # A randomization failure is recorded under the original planned seed.\n   failed=te.Experiment(root,run,name='forced_failure_fixture',batch_cells=3);failed.prepare_precision(scopes)\n   with patch.object(failed,'graph',side_effect=RuntimeError('synthetic swap failure')):\n    status=failed.model(s,'M4',1)\n   assert status['status']=='FAILED' and status['seed_index']==1\n   assert (failed.scope_dir(s)/'outer/models/M4_seed001/failure.json').exists()\n   assert failed.model(s,'M4',1)==status\n   # Corrupted snapshot/cache must be rejected, never silently regenerated.\n   p=ex.scope_dir(s)/'outer/models/M0/model.npz';p.write_bytes(b'corrupt');reject(lambda:ex.model(s,'M0'))\n  return {'status':'PASS','checks':['Full frozen grids and all 20 M3/M4 seeds on one synthetic scope','Real numerical gate uses T/V, no B; registry/ledger precedes final scoring','M0/M1 -> M2 -> M3 -> M4 order and exact common-I hashes','M3 matched trace; all seed rows retained','Final score agrees with dense count oracle incl zero-library B','Checkpoint retry, old lock unchanged, premature expansion/corruption rejected','Forced randomization failure retained under original seed; no replacement'],'n_scores':len(combined),'B_shard_reads':len(reads)}\n\ndef amendment_tests():\n with tempfile.TemporaryDirectory() as tmp:\n  root=Path(tmp);run=fixture(root);parent=te.Experiment(root,run);scope=parent.scopes[3]\n  original_fit=parent.fit\n  def forced_probe(r,p,method,c,tol,allow=False):\n   if method=='M2' and c['alpha']==100 and c['beta']==100 and c['lambda_rule']=='q0.5':raise RuntimeError('Adaptive maximum iterations reached')\n   return original_fit(r,p,method,c,tol,allow)\n  with contextlib.redirect_stdout(io.StringIO()),patch.object(parent,'fit',side_effect=forced_probe):reject(lambda:parent.precision_gate(scope))\n  saved={str(p):te.sha(p) for folder in [parent.old,parent.out,root/'configs'] for p in folder.rglob('*') if p.is_file()}\n  amended=te.Experiment(root,run,name='task2_ordered_controls_num_v2',numerical_amendment='adaptive_budget_1000_3000_v1')\n  def differences(a,b,path=''):\n   if isinstance(a,dict):return sum((differences(a[k],b[k],path+'.'+k) for k in a),[])\n   return [path] if a!=b else []\n  assert set(differences(parent.grid,amended.grid))=={'.precision.standard.adaptive_max_outer','.precision.tight.adaptive_max_outer','.source_coarse.convergence.adaptive_max_outer'}\n  assert amended.contract['numerical_amendment']['new_final_B_used_for_amendment'] is False\n  assert te.js(amended.out/'effective_parameter_grid.json')==amended.grid\n  seen=[];fit=amended.fit\n  def tracked(r,p,m,c,t,allow=False):seen.append(t['adaptive_max_outer']);return fit(r,p,m,c,t,allow)\n  with contextlib.redirect_stdout(io.StringIO()),patch.object(amended,'fit',side_effect=tracked):\n   amended.prepare_precision([scope])\n   tables=[amended.run_stage(stage,[scope],'amendment_fixture') for stage in te.STAGES]\n  table=pd.concat(tables,ignore_index=True);assert len(table)==43 and table.status.eq('OK').all()\n  assert set(seen)=={1000,3000},set(seen)\n  assert all(te.sha(Path(p))==h for p,h in saved.items())\n  legacy_zip=parent.review_bundle();legacy_bytes=legacy_zip.read_bytes();new_zip=amended.review_bundle()\n  assert new_zip!=legacy_zip and legacy_zip.read_bytes()==legacy_bytes\n  import zipfile\n  with zipfile.ZipFile(new_zip) as bundle:\n   assert json.loads(bundle.read('effective_parameter_grid.json'))==amended.grid\n   assert json.loads(bundle.read('configs/parameter_grid.yaml'))==parent.grid\n  again=te.Experiment(root,run,name='task2_ordered_controls_num_v2',numerical_amendment='adaptive_budget_1000_3000_v1');assert again.contract==amended.contract\n  reject(lambda:te.Experiment(root,run,name='bad_amendment',numerical_amendment='arbitrary'))\n  reject(lambda:te.Experiment(root,run,numerical_amendment='adaptive_budget_1000_3000_v1'))\n  reject(lambda:te.numerical_grid_amendment(amended.grid))\n  te.save(parent.out/'scopes/scope_03/inner/tuning/M2/started.json',{'status':'started'})\n  reject(lambda:te.Experiment(root,run,name='bad_searched_parent',numerical_amendment='adaptive_budget_1000_3000_v1'))\n  (parent.out/'scopes/scope_03/inner/tuning/M2/started.json').unlink()\n  (parent.out/'scopes/scope_03/inner/tuning/M2').rmdir()\n  te.save(parent.out/'scores/fixture/fixed/target_access_ledger.json',{'status':'registered'})\n  reject(lambda:te.Experiment(root,run,name='bad_scored_parent',numerical_amendment='adaptive_budget_1000_3000_v1'))\n  return {'status':'PASS','checks':['Exactly three numerical budget fields changed; tolerances/grid/seeds identical','New real-gate path and complete 43-row synthetic stage sequence use amended budgets','Original failure/source protocol/configs unchanged','Effective grid frozen; same identity retry; invalid version, old run, repeated amendment and scored parent rejected'],'n_scores':len(table)}\n\n# Imported helpers are provided by the embedded main test source.\ndef accuracy_v3_tests():\n with tempfile.TemporaryDirectory() as tmp:\n  root=Path(tmp);run=fixture(root);original=te.Experiment(root,run);sf=original.scopes[3];fit=original.fit\n  def forced(r,p,m,c,t,allow=False):\n   if m=='M2' and c['alpha']==100 and c['beta']==100 and c['lambda_rule']=='q0.5':raise RuntimeError('Adaptive maximum iterations reached')\n   return fit(r,p,m,c,t,allow)\n  with contextlib.redirect_stdout(io.StringIO()),patch.object(original,'fit',side_effect=forced):reject(lambda:original.precision_gate(sf))\n  parent=te.Experiment(root,run,name='task2_ordered_controls_num_v2',numerical_amendment='adaptive_budget_1000_3000_v1')\n  cmp=ps.precision_comparison\n  def disagreement(*args,**kwargs):\n   result=cmp(*args,**kwargs)\n   if len(args)>4 and args[4] is not None:result.update(status='FAIL',absolute_s_max=2e-5)\n   return result\n  with contextlib.redirect_stdout(io.StringIO()),patch.object(ps,'precision_comparison',side_effect=disagreement):reject(lambda:parent.precision_gate(parent.scopes[7]))\n  grid=te.numerical_accuracy_amendment(original.grid);folder=parent.out/'numerical_diagnostics/accuracy_v3_candidate'\n  plan={'role':'development_only_diagnostic','final_B_used':False,'authorizes_search':False,'standard':grid['precision']['standard'],'tight':grid['precision']['tight'],'thresholds':grid['precision']['thresholds'],'scope_ids':[s['scope_id'] for s in parent.first_four()],'methods':['M0','M1','M2','M4'],'M4_seed_index':0,'probe_parameters':grid['precision']['probe_parameters'],'engine_sha256':parent.contract['engine_sha256']}\n  te.save(folder/'diagnostic_plan.json',plan);rows=[];load=sp.load_npz\n  def forbid_B(path,*args,**kwargs):\n   assert Path(path).name!='B_target.npz','Diagnostic read B'\n   return load(path,*args,**kwargs)\n  with patch.object(sp,'load_npz',side_effect=forbid_B):\n   for s in parent.first_four():\n    r=parent.representation(s,'inner');base=parent.base_ppi(s,'inner');g4=parent.graph(s,'inner','M4',0);loss=parent.loss(s,r)\n    for method in plan['methods']:\n     for c in plan['probe_parameters']:\n      for rule in (['q0.5','high'] if method=='M2' and c['beta']>0 else ['fixed_limit']):\n       par={**c,'beta':0. if method=='M0' else c['beta'],'lambda_rule':rule};p=g4 if method=='M4' else base\n       x=parent.fit(r,p,method,par,plan['standard'],True);y=parent.fit(r,p,method,par,plan['tight'],True)\n       comp=cmp(x['prediction'],y['prediction'],loss,plan['thresholds'],x['s'] if method=='M2' else None,y['s'] if method=='M2' else None)\n       assert comp['status']=='PASS',comp\n       rows.append({**s,'method':method,**par,'standard_iterations':x['iterations'],'tight_iterations':y['iterations'],**comp})\n  table=pd.DataFrame(rows);table.to_csv(folder/'probes.csv',index=False);te.save(folder/'summary.json',{'status':'PASS','n_probes':len(table),'n_failed':0,'final_B_used':False,'authorizes_search':False})\n  assert len(table)==152\n  evidence=te.verify_accuracy_evidence(parent.out,grid,parent.scopes,parent.contract);assert evidence['fresh_real_gates_required']\n  saved={str(p):te.sha(p) for directory in [original.old,original.out,parent.out,root/'configs'] for p in directory.rglob('*') if p.is_file()}\n  amended=te.Experiment(root,run,name='task2_ordered_controls_num_v3',numerical_amendment='adaptive_accuracy_v3');assert amended.grid==grid\n  assert amended.grid['coarse']==original.grid['coarse'] and amended.grid['fine']==original.grid['fine'] and amended.grid['precision']['thresholds']==original.grid['precision']['thresholds']\n  seen=[];fit=amended.fit\n  def tracked(r,p,m,c,t,allow=False):seen.append((t['cg_rtol'],t['adaptive_x_rtol'],t['adaptive_s_atol'],t['adaptive_max_outer']));return fit(r,p,m,c,t,allow)\n  with contextlib.redirect_stdout(io.StringIO()),patch.object(amended,'fit',side_effect=tracked):\n   scopes=[amended.first_four()[0]];amended.prepare_precision(scopes)\n   scores=pd.concat([amended.run_stage(stage,scopes,'v3_fixture') for stage in te.STAGES],ignore_index=True)\n  assert len(scores)==43 and scores.status.eq('OK').all()\n  assert set(seen)=={(1e-10,1e-9,1e-8,1000),(1e-12,1e-11,1e-10,3000)}\n  assert all(te.sha(Path(p))==h for p,h in saved.items())\n  repeated=te.Experiment(root,run,name='task2_ordered_controls_num_v3',numerical_amendment='adaptive_accuracy_v3');assert repeated.contract==amended.contract\n  original_csv=(folder/'probes.csv').read_bytes()\n  for corrupted in [table.iloc[:-1],pd.concat([table.iloc[:-1],table.iloc[:1]])]:\n   corrupted.to_csv(folder/'probes.csv',index=False);reject(lambda:te.verify_accuracy_evidence(parent.out,grid,parent.scopes,parent.contract))\n  bad=table.copy();bad.loc[bad.method.eq('M2'),'absolute_s_max']=2e-5;bad.to_csv(folder/'probes.csv',index=False);reject(lambda:te.verify_accuracy_evidence(parent.out,grid,parent.scopes,parent.contract))\n  (folder/'probes.csv').write_bytes(original_csv)\n  original_plan=(folder/'diagnostic_plan.json').read_bytes();badplan=json.loads(json.dumps(plan));badplan['thresholds']['absolute_s_max']=1e-4;te.save(folder/'diagnostic_plan.json',badplan);reject(lambda:te.verify_accuracy_evidence(parent.out,grid,parent.scopes,parent.contract));(folder/'diagnostic_plan.json').write_bytes(original_plan)\n  oldzip=parent.review_bundle();oldbytes=oldzip.read_bytes();newzip=amended.review_bundle();assert oldzip!=newzip and oldzip.read_bytes()==oldbytes\n  return {'status':'PASS','checks':['All 152 real synthetic diagnostic probes, no B read','Evidence coverage, duplicates, precision failures and changed plan rejected','Exact reviewed v3 rules used in precision/search/refit/replay and all43 method/seed rows','Original runs/protocol/configs unchanged; retry contract and ZIP protection'],'n_diagnostic_probes':len(table),'n_scores':len(scores)}\n\ndef run_tests():return {'mathematical':mathematical_tests(),'integration':integration_tests(),'numerical_amendment':amendment_tests(),'accuracy_v3':accuracy_v3_tests()}\nif __name__=='__main__':print(json.dumps(run_tests(),indent=2))\n"
test_ns={"__name__":"task2_experiments_tests","__file__":str(ROOT/"tests/test_task2_experiments.py")}
exec(compile(TEST_SOURCE,"embedded_task2_experiments_tests.py","exec"),test_ns)
TEST_REPORT=test_ns["run_tests"]()
assert all(v["status"]=="PASS" for v in TEST_REPORT.values())
print(json.dumps(TEST_REPORT,ensure_ascii=False,indent=2))


{
  "mathematical": {
    "status": "PASS",
    "checks": [
      "CG/warm CG independent Kronecker oracle and fast Laplacian identity",
      "read-only NumPy views: nullable metadata, inner/outer, no matches, empty frame, input preservation",
      "development sufficient statistics vs dense all-zero/mixed-floor/q oracle"
    ]
  },
  "integration": {
    "status": "PASS",
    "checks": [
      "Full frozen grids and all 20 M3/M4 seeds on one synthetic scope",
      "Real numerical gate uses T/V, no B; registry/ledger precedes final scoring",
      "M0/M1 -> M2 -> M3 -> M4 order and exact common-I hashes",
      "M3 matched trace; all seed rows retained",
      "Final score agrees with dense count oracle incl zero-library B",
      "Checkpoint retry, old lock unchanged, premature expansion/corruption rejected",
      "Forced randomization failure retained under original seed; no replacement"
    ],
    "n_scores": 43,
    "B_shard_reads": 5
  },
  "numerical_amendment": {
    "status

In [5]:
EX=te.Experiment(ROOT,RUN,PROTOCOL,EXPERIMENT_NAME,BATCH_CELLS,
                 numerical_amendment=NUMERICAL_AMENDMENT)
FOUR_SCOPES=EX.first_four()
print("Actual first round scopes：")
print(__import__("pandas").DataFrame(FOUR_SCOPES).to_string(index=False))
te.save(EX.out/"synthetic_test_report.json",TEST_REPORT)
def run_with_review(call):
    try:return call()
    except (Exception,KeyboardInterrupt):
        try:print("Partial verification package saved：",EX.review_bundle())
        except Exception as e:print("Failed to generate verification package, please keep the original traceback:",repr(e))
        raise
def preview(frame):
    cols=["cell_line","drug","replicate","method","seed_index","status","mean_poisson_deviance","normalized_mse"]
    print(frame.reindex(columns=cols).to_string(index=False))


Actual first round scopes：
 scope_id cell_line       drug   replicate  n_cells   n_pairs
        2      A172  Lapatinib replicate_1    47258 118145000
        3      A172  Lapatinib replicate_2    48061 120152500
        6      A172 Trametinib replicate_1    13723  34307500
        7      A172 Trametinib replicate_2    19639  49097500
       14      T98G Nintedanib replicate_1    31554  78885000
       15      T98G Nintedanib replicate_2    17517  43792500
       28     U87MG    ZSTK474 replicate_1    17235  43087500
       29     U87MG    ZSTK474 replicate_2    33272  83180000


In [6]:
run_with_review(lambda:EX.prepare_precision(FOUR_SCOPES))
print("All first-round real T/V precision gates have passed.")


Partial verification package saved： /home/featurize/work/biological_prior_validation/results/summary/task2_step3_review_bundle_task2_ordered_controls_num_v2.zip


RuntimeError: Existing real precision gate failed; new numerical version required

In [ ]:
FIXED=run_with_review(lambda:EX.run_stage("fixed",FOUR_SCOPES))
preview(FIXED)


In [ ]:
ADAPTIVE=run_with_review(lambda:EX.run_stage("adaptive",FOUR_SCOPES))
preview(ADAPTIVE)


In [ ]:
MATCHED=run_with_review(lambda:EX.run_stage("matched",FOUR_SCOPES))
preview(MATCHED)


In [ ]:
REWIRED=run_with_review(lambda:EX.run_stage("rewired",FOUR_SCOPES))
preview(REWIRED)


In [ ]:
FIRST_FOUR,STABILITY=run_with_review(EX.finalize_first_four)
print(json.dumps(STABILITY,ensure_ascii=False,indent=2))
print("Save complete groups and seed scores:",EX.out/"first_four_all_scores.csv")


In [ ]:
ALL15=run_with_review(EX.run_all15) if RUN_ALL15 else None
if ALL15 is None:
    print("This time it stopped at four exploratory contexts, and the complete extended code is ready. Please send back the verification package first.")
else:
    print("All 15 contexts have been saved:",EX.out/"all15_all_scores.csv")

In [ ]:
EX.check_lock()
RAW_DIR=ROOT/"results/raw"/EXPERIMENT_NAME
RAW_DIR.mkdir(parents=True,exist_ok=True)
for filename in ["first_four_all_scores.csv","all15_all_scores.csv"]:
    source_path=EX.out/filename
    if source_path.exists():
        destination=RAW_DIR/filename
        if destination.exists() and te.sha(destination)!=te.sha(source_path):
            raise ValueError(f"Existing raw results are different; overwrite is refused.：{destination}")
        if not destination.exists():destination.write_bytes(source_path.read_bytes())
mc.freeze(ROOT/"configs"/EXPERIMENT_NAME/"experiment_contract.json",EX.contract)
REVIEW_ZIP=EX.review_bundle()
print("Please return：",REVIEW_ZIP)
print("Original result：",RAW_DIR)
print("Keep the old Step 5 lock consistent：",EX.old_lock)


In [ ]:
import json
import pandas as pd
from pathlib import Path

# Requires the V2 notebook's cell 5 objects (EX, FOUR_SCOPES, te, mc, ps)
parent = EX.out
v2_gate = parent / "scopes/scope_07/precision/gate.json"
v2_probes = parent / "scopes/scope_07/precision/probes.csv"
if not v2_gate.is_file() or not v2_probes.is_file():
    raise FileNotFoundError("V2 scope_07 gate/probes are required")
gate = te.js(v2_gate)
old_probes = pd.read_csv(v2_probes)
failed = old_probes[
    old_probes.method.eq("M2")
    & old_probes.alpha.eq(100)
    & old_probes.beta.eq(100)
    & old_probes.lambda_rule.eq("q0.5")
    & old_probes.status.eq("FAIL")
]
if gate.get("status") != "FAIL" or gate.get("final_B_used") is not False:
    raise ValueError("V2 scope_07 must be a development-only FAIL")
if not len(failed) or not failed.absolute_s_max.gt(
    EX.grid["precision"]["thresholds"]["absolute_s_max"]
).all():
    raise ValueError("V2 failure does not match the documented s precision discrepancy")

# Load the original frozen parameter grid
base_grid = te.js(EX.gridfolder / "parameter_grid.yaml")

# Verify that the current experiment uses the expected V2 rules
expected_v2_grid = te.numerical_grid_amendment(base_grid)

if expected_v2_grid != EX.grid:
    raise ValueError(
        "Current V2 grid does not match the frozen original configuration"
    )

# Construct V3 rules from the original frozen grid
v3_grid = te.numerical_accuracy_amendment(base_grid)

print("V2 configuration verified.")
print("V3 configuration constructed successfully.")
selected = [
    s for line, drug in mc.FOUR for rep in mc.REPS
    for s in FOUR_SCOPES
    if (s["cell_line"], s["drug"], s["replicate"]) == (line, drug, rep)
]
plan = {
    "role": "development_only_diagnostic",
    "final_B_used": False,
    "authorizes_search": False,
    "standard": v3_grid["precision"]["standard"],
    "tight": v3_grid["precision"]["tight"],
    "thresholds": v3_grid["precision"]["thresholds"],
    "scope_ids": [s["scope_id"] for s in selected],
    "methods": ["M0", "M1", "M2", "M4"],
    "M4_seed_index": 0,
    "probe_parameters": v3_grid["precision"]["probe_parameters"],
    "engine_sha256": te.js(parent / "experiment_contract.json")["engine_sha256"],
}
if len(selected) != 8:
    raise ValueError(f"Expected all 8 first-four scopes; got {len(selected)}")

rows=[]
for scope in selected:
    r=EX.representation(scope,"inner")
    base=EX.base_ppi(scope,"inner")
    loss=EX.loss(scope,r)
    m4=EX.graph(scope,"inner","M4",0)
    for method in plan["methods"]:
        for params in plan["probe_parameters"]:
            beta=0.0 if method=="M0" else float(params["beta"])
            rules=( ["q0.5","high"] if method=="M2" and beta>0
                    else ["fixed_limit"] )
            for rule in rules:
                c={"alpha":float(params["alpha"]),"beta":beta,"lambda_rule":rule}
                p=m4 if method=="M4" else base
                row={"scope_id":scope["scope_id"],"cell_line":scope["cell_line"],
                     "drug":scope["drug"],"replicate":scope["replicate"],
                     "n_cells":scope["n_cells"],"n_pairs":scope["n_pairs"],
                     "method":method,**c}
                try:
                    a=EX.fit(r,p,method,c,v3_grid["precision"]["standard"],True)
                    b=EX.fit(r,p,method,c,v3_grid["precision"]["tight"],True)
                    if a.get("status")!="OK" or b.get("status")!="OK":
                        raise RuntimeError("Precision probe did not return OK")
                    comp=ps.precision_comparison(
                        a["prediction"],b["prediction"],loss,
                        v3_grid["precision"]["thresholds"],
                        a["s"] if method=="M2" else None,
                        b["s"] if method=="M2" else None)
                    row.update(comp)
                    row["standard_tau"]=a["tau"];row["tight_tau"]=b["tau"]
                    row["standard_iterations"]=a["iterations"]
                    row["tight_iterations"]=b["iterations"]
                    row["degenerate_training_fixed_limit"]=a.get("degenerate_fixed_limit",False)
                except (RuntimeError,ValueError,FloatingPointError) as e:
                    row.update(status="FAIL",failure_reason=str(e),
                               standard_iterations=None,tight_iterations=None)
                rows.append(row)

table=pd.DataFrame(rows)
folder=parent/"numerical_diagnostics/accuracy_v3_candidate"
folder.mkdir(parents=True,exist_ok=True)
table.to_csv(folder/"probes.csv",index=False)
all_pass=(len(table)==152 and table.status.eq("PASS").all())
summary={"status":"PASS" if all_pass else "FAIL","n_probes":len(table),
         "n_failed":int(table.status.ne("PASS").sum()),
         "final_B_used":False,"authorizes_search":False}
te.save(folder/"diagnostic_plan.json",plan)
te.save(folder/"summary.json",summary)
te.receipt(folder,["diagnostic_plan.json","probes.csv","summary.json"],
           {"role":"development_only_diagnostic","parent_experiment":str(parent)})
print(json.dumps(summary,indent=2))
print("Evidence:",folder)
if not all_pass:
    raise RuntimeError("V3 diagnostic did not pass; preserve evidence and stop")
    
print("V3 standard:")
print(v3_grid["precision"]["standard"])

print("\nV3 tight:")
print(v3_grid["precision"]["tight"])

print("\nV3 source coarse:")
print(v3_grid["source_coarse"]["convergence"])